# NEM facility monitoring and MQTT replay

This notebook builds the local data cache, MQTT publisher, DuckDB integration, and Dash Leaflet subscriber and dashboard.


## 1. Configuration

The selected time range covers one full week in May 2026. OpenElectricity expects date filters in network-local time without a timezone suffix.
Full retrieval is disabled by default to avoid unnecessary API calls.


In [ ]:
from __future__ import annotations

import asyncio
import math
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Iterable

import pandas as pd
from IPython.display import display


PROJECT_ROOT = Path("..") if Path.cwd().name == "streaming_pipeline" else Path(".")
STREAM_DIR = PROJECT_ROOT / "data" / "streaming"
RAW_DIR = STREAM_DIR / "Task1_output"
RAW_DIR.mkdir(parents=True, exist_ok=True)


@dataclass(frozen=True)
class RetrievalConfig:
    """Configuration values used by the retrieval workflow."""

    network_code: str = "NEM"
    interval: str = "5m"
    date_start: str = "2026-05-01T00:00:00"
    date_end: str = "2026-05-08T00:00:00"
    facility_batch_size: int = 20
    request_limit_per_day: int = 500


CONFIG = RetrievalConfig()

# Keep full retrieval off unless a new API download is required.
RUN_SMOKE_TEST = False
RUN_FULL_RETRIEVAL = False  # set True only if Task1_output/ is missing or needs refresh

print(CONFIG)

## 2. Environment loading

Set `OPENELECTRICITY_API_KEY` in the process environment or the repository-root `.env` file.


In [ ]:
from dotenv import load_dotenv

load_dotenv(PROJECT_ROOT / ".env")


def require_api_key() -> str:
    """Return the OpenElectricity credential without displaying it."""
    api_key = os.getenv("OPENELECTRICITY_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("Set OPENELECTRICITY_API_KEY in the environment or the repository-root .env file.")
    return api_key


## 3. SDK Imports and Client Creation

This section imports the OpenElectricity SDK and creates an async client. The async client is required because Jupyter already runs an event loop.


In [ ]:
def import_openelectricity_sdk() -> dict[str, Any]:
    """Import SDK objects in one place."""
    try:
        from openelectricity import AsyncOEClient
        from openelectricity.types import DataMetric, MarketMetric
    except ModuleNotFoundError as exc:
        raise ModuleNotFoundError(
            "Install the official SDK first: pip install openelectricity"
        ) from exc

    try:
        from openelectricity.exceptions import OpenElectricityError
    except Exception:
        OpenElectricityError = Exception

    return {
        "AsyncOEClient": AsyncOEClient,
        "DataMetric": DataMetric,
        "MarketMetric": MarketMetric,
        "OpenElectricityError": OpenElectricityError,
    }
def get_async_oe_client():
    """Create the async OpenElectricity client used in this notebook."""
    sdk = import_openelectricity_sdk()
    api_key = require_api_key()
    return sdk["AsyncOEClient"](api_key=api_key)


def verify_async_client_context_manager() -> None:
    """Check that the installed async client supports async with."""
    sdk = import_openelectricity_sdk()
    client_cls = sdk["AsyncOEClient"]
    if not hasattr(client_cls, "__aenter__") or not hasattr(client_cls, "__aexit__"):
        raise RuntimeError(
            "Installed AsyncOEClient does not support 'async with'. Use an SDK version with async context-manager support."
        )


print("SDK import function ready.")

## 4. Request Counting and Error Handling

This section counts API calls and handles common API errors. Rate limits and temporary network errors are retried with controlled delays.


In [ ]:
request_counter = {"count": 0}
LAST_API_RESPONSES: dict[str, Any] = {}


def get_error_status(exc: Exception) -> int | None:
    """Extract an HTTP status code from an SDK exception when available."""
    for attr in ("status", "status_code"):
        value = getattr(exc, attr, None)
        if isinstance(value, int):
            return value
    response = getattr(exc, "response", None)
    value = getattr(response, "status_code", None)
    return value if isinstance(value, int) else None


def is_auth_error(exc: Exception) -> bool:
    """Detect authentication and permission errors."""
    message = str(exc).lower()
    status = get_error_status(exc)
    return status in {401, 403} or "unauthorized" in message or "forbidden" in message


def is_rate_limit_error(exc: Exception) -> bool:
    """Detect rate-limit errors."""
    message = str(exc).lower()
    status = get_error_status(exc)
    return status == 429 or "429" in message or "rate limit" in message


def is_transient_network_error(exc: Exception) -> bool:
    """Detect temporary network errors."""
    message = str(exc).lower()
    error_name = type(exc).__name__.lower()
    return any(
        token in message or token in error_name
        for token in ["connect", "timeout", "temporarily", "network", "clientconnectorerror"]
    )

async def call_api_with_retry_async(callable_obj, *, label: str, retry_after_seconds: int = 60, network_retries: int = 3):
    """Call an async API method with retries and clear error handling."""
    max_attempts = max(2, network_retries)
    for attempt in range(1, max_attempts + 1):
        try:
            request_counter["count"] += 1
            return await callable_obj()
        except Exception as exc:
            if is_auth_error(exc):
                raise RuntimeError(
                    "OpenElectricity authentication failed. Check that the API key is complete, active, and copied without extra text."
                ) from exc
            if is_rate_limit_error(exc) and attempt == 1:
                print(f"Rate limit during {label}; waiting {retry_after_seconds}s then retrying once.")
                await asyncio.sleep(retry_after_seconds)
                continue
            if is_transient_network_error(exc) and attempt < max_attempts:
                print(f"Network error during {label}; retrying in 5s ({attempt}/{max_attempts}).")
                await asyncio.sleep(5)
                continue
            if is_transient_network_error(exc):
                raise RuntimeError(
                    f"OpenElectricity network request failed for {label}. Check VPN, proxy, firewall, or internet connection."
                ) from exc
            print(f"WARNING: API call failed for {label}: {exc}")
            return None


def chunks(values: list[str], size: int) -> Iterable[list[str]]:
    """Split facility codes into fixed-size batches."""
    for start in range(0, len(values), size):
        yield values[start : start + size]


print("Request utilities ready.")

## 5. Response Normalisation Helpers

OpenElectricity returns typed SDK objects. These helpers convert them into pandas DataFrames with stable column names.


In [ ]:
def enum_value(value: Any) -> Any:
    """Return the plain value from enum-like SDK objects."""
    return getattr(value, "value", value)


def first_attr(obj: Any, names: list[str], default: Any = None) -> Any:
    """Read the first available field from a dict or SDK object."""
    if obj is None:
        return default
    for name in names:
        if isinstance(obj, dict) and name in obj:
            return obj[name]
        if hasattr(obj, name):
            return getattr(obj, name)
    return default


def nested_location_value(facility: Any, key: str) -> Any:
    """Read latitude or longitude from facility location fields."""
    direct_names = [key, "lat"] if key == "latitude" else [key, "lng", "lon"]
    direct_value = first_attr(facility, direct_names)
    if direct_value is not None:
        return direct_value
    location = first_attr(facility, ["location"])
    location_names = ["lat"] if key == "latitude" else ["lng", "lon"]
    return first_attr(location, location_names)


def model_dump_if_available(obj: Any) -> dict[str, Any]:
    """Convert SDK model objects to dictionaries when possible."""
    if obj is None:
        return {}
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "model_dump"):
        return obj.model_dump()
    return {}


FACILITY_COLUMNS = [
    "facility_code",
    "facility_name",
    "unit_code",
    "network_id",
    "network_region",
    "fueltech_id",
    "status",
    "latitude",
    "longitude",
    "capacity_mw",
]


def facilities_to_dataframe(response: Any) -> pd.DataFrame:
    """Convert facility metadata into one row per facility unit."""
    rows: list[dict[str, Any]] = []

    for facility in getattr(response, "data", []) or []:
        units = getattr(facility, "units", None) or [None]
        for unit in units:
            rows.append(
                {
                    "facility_code": first_attr(facility, ["code", "facility_code", "id"]),
                    "facility_name": first_attr(facility, ["name", "facility_name"]),
                    "unit_code": first_attr(unit, ["code", "unit_code", "id"]) if unit else None,
                    "network_id": enum_value(first_attr(facility, ["network_id", "network_code"])),
                    "network_region": enum_value(first_attr(facility, ["network_region", "region"])),
                    "fueltech_id": enum_value(
                        first_attr(unit, ["fueltech_id", "fueltech"]) if unit else first_attr(facility, ["fueltech_id", "fueltech"])
                    ),
                    "status": enum_value(
                        first_attr(unit, ["status_id", "status"]) if unit else first_attr(facility, ["status_id", "status"])
                    ),
                    "latitude": nested_location_value(facility, "latitude"),
                    "longitude": nested_location_value(facility, "longitude"),
                    "capacity_mw": first_attr(unit, ["capacity_mw", "capacity", "capacity_registered", "capacity_maximum"]) if unit else first_attr(facility, ["capacity_mw", "capacity"]),
                }
            )

    return pd.DataFrame(rows, columns=FACILITY_COLUMNS)


def timeseries_response_to_dataframe(response: Any) -> pd.DataFrame:
    """Convert a time-series response into long-format rows."""
    rows: list[dict[str, Any]] = []

    for series in getattr(response, "data", []) or []:
        metric = enum_value(getattr(series, "metric", None))
        unit = getattr(series, "unit", None)

        for result in getattr(series, "results", []) or []:
            result_name = first_attr(result, ["name", "code", "id"])
            columns = first_attr(result, ["columns"])
            columns_dict = model_dump_if_available(columns)
            unit_code = first_attr(result, ["unit_code", "unit"], columns_dict.get("unit_code"))
            region = first_attr(result, ["network_region", "region"], columns_dict.get("network_region"))
            # Facility data uses result.name as the facility or series identifier.
            # Unit code is stored separately to avoid mixing facility and unit identifiers.
            facility_code = first_attr(result, ["facility_code", "facility"], result_name)

            for point in getattr(result, "data", []) or []:
                rows.append(
                    {
                        "timestamp": getattr(point, "timestamp", None),
                        "facility_code": facility_code,
                        "unit_code": unit_code,
                        "network_region": enum_value(region),
                        "series_name": result_name,
                        "metric": metric,
                        "value": getattr(point, "value", None),
                        "unit": unit,
                    }
                )

    return pd.DataFrame(rows)


def debug_timeseries_structure(response: Any, max_series: int = 2, max_results: int = 2) -> None:
    """Print a compact response sample for field-path validation."""
    for series_index, series in enumerate((getattr(response, "data", []) or [])[:max_series], start=1):
        print(f"Series {series_index}: metric={enum_value(getattr(series, 'metric', None))}, groupings={getattr(series, 'groupings', None)}")
        for result_index, result in enumerate((getattr(series, "results", []) or [])[:max_results], start=1):
            columns = model_dump_if_available(first_attr(result, ["columns"]))
            first_point = (getattr(result, "data", []) or [None])[0]
            print(
                f"  Result {result_index}: name={first_attr(result, ['name'])}, "
                f"columns={columns}, first_point={first_point}"
            )


print("Normalisation helpers ready.")

## 6. Retrieval Functions

These functions retrieve facility metadata, facility time-series data, and market data. They form the main Task 1 retrieval workflow.


In [ ]:
def parse_datetime(value: str) -> datetime:
    """Parse a network-local ISO timestamp."""
    return datetime.fromisoformat(value)


async def fetch_facilities(client, config: RetrievalConfig = CONFIG) -> pd.DataFrame:
    """Fetch NEM facility metadata and keep operating units."""
    async def api_call():
        # Fetch all NEM facilities first, then apply status filtering locally.
        return await client.get_facilities(network_id=[config.network_code])

    response = await call_api_with_retry_async(api_call, label="facility metadata")
    if response is None:
        return pd.DataFrame()
    LAST_API_RESPONSES["facilities"] = response

    raw_count = len(getattr(response, "data", []) or [])
    facilities = facilities_to_dataframe(response)
    print(f"Fetched {raw_count} facilities from API; parsed {len(facilities)} facility-unit rows.")
    if facilities.empty:
        print("WARNING: facility metadata response was empty.")
        return facilities

    facilities = facilities.dropna(subset=["facility_code"]).drop_duplicates()
    if "status" in facilities.columns and facilities["status"].notna().any():
        operating_mask = facilities["status"].astype(str).str.strip().str.lower() == "operating"
        if operating_mask.any():
            facilities = facilities.loc[operating_mask].copy()
    return facilities.sort_values(["facility_code", "unit_code"], na_position="last").reset_index(drop=True)


async def fetch_facility_timeseries(
    client,
    facility_codes: list[str],
    config: RetrievalConfig = CONFIG,
) -> pd.DataFrame:
    """Fetch 5-minute power and emissions data by facility batch."""
    # SDK 0.10.1 uses network_code here, while get_facilities uses network_id.
    # The smoke test verifies these SDK parameter names.
    sdk = import_openelectricity_sdk()
    DataMetric = sdk["DataMetric"]

    start = parse_datetime(config.date_start)
    end = parse_datetime(config.date_end)
    frames: list[pd.DataFrame] = []

    for index, batch in enumerate(chunks(facility_codes, config.facility_batch_size), start=1):
        label = f"facility time-series batch {index} ({len(batch)} facilities)"

        async def api_call(batch=batch):
            return await client.get_facility_data(
                network_code=config.network_code,
                facility_code=batch,
                metrics=[DataMetric.POWER, DataMetric.EMISSIONS],
                interval=config.interval,
                date_start=start,
                date_end=end,
            )

        response = await call_api_with_retry_async(api_call, label=label)
        if response is None:
            continue
        LAST_API_RESPONSES["facility_timeseries"] = response

        df = timeseries_response_to_dataframe(response)
        if df.empty:
            print(f"WARNING: {label} returned no rows.")
            continue
        frames.append(df)

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


async def fetch_market_data(client, config: RetrievalConfig = CONFIG) -> pd.DataFrame:
    """Fetch market price and demand data for dashboard use."""
    # SDK 0.10.1 supports network_code and primary_grouping here.
    sdk = import_openelectricity_sdk()
    MarketMetric = sdk["MarketMetric"]
    start = parse_datetime(config.date_start)
    end = parse_datetime(config.date_end)

    async def api_call():
        return await client.get_market(
            network_code=config.network_code,
            metrics=[MarketMetric.PRICE, MarketMetric.DEMAND],
            interval=config.interval,
            date_start=start,
            date_end=end,
            primary_grouping="network_region",
        )

    response = await call_api_with_retry_async(api_call, label="market price and demand")
    if response is not None:
        LAST_API_RESPONSES["market"] = response
    market_df = timeseries_response_to_dataframe(response) if response is not None else pd.DataFrame()
    if not market_df.empty:
        # Market result names identify regions or networks, not facilities.
        market_df = market_df.rename(columns={"facility_code": "market_region"})
        if "network_region" in market_df.columns:
            market_df["network_region"] = market_df["network_region"].fillna(market_df["market_region"])
    return market_df


print("Retrieval functions ready.")

## 7. Request-Limit Calculation

This section estimates the number of API requests before full retrieval. This keeps the run below the 500 requests/day limit.


In [ ]:
def estimate_initial_requests(num_facilities: int, config: RetrievalConfig = CONFIG) -> dict[str, int]:
    """Estimate API requests for the initial one-week retrieval."""
    facility_requests = math.ceil(num_facilities / config.facility_batch_size)
    return {
        "metadata_requests": 1,
        "facility_timeseries_requests": facility_requests,
        "market_requests": 1,
        "estimated_total": 1 + facility_requests + 1,
        "daily_limit": config.request_limit_per_day,
    }


# Example estimate. The full run prints the real estimate.
estimate_initial_requests(num_facilities=300)

## 8. Smoke Test

The smoke test checks the API key, SDK methods, response structure, batching logic, and DataFrame conversion on a small sample.


In [ ]:
# ── Task 1 Smoke Test ────────────────────────────────────────────────────────
# Run this cell FIRST to verify: API key validity, SDK context-manager support,
# response structure, and DataFrame conversion.
# Uses a 1-hour window for two facilities only (minimal API quota consumed).
# Enable: set RUN_SMOKE_TEST = True in the Configuration cell (Section 1).

if RUN_SMOKE_TEST:
    request_counter["count"] = 0
    verify_async_client_context_manager()
    print("AsyncOEClient context-manager support: OK")

    async with get_async_oe_client() as client:
        facilities_smoke = await fetch_facilities(client)
        display(facilities_smoke.head())

        if facilities_smoke.empty:
            print("fetch_facilities() returned an empty DataFrame. Running a raw SDK diagnostic now...")
            try:
                raw_response = await client.get_facilities(network_id=[CONFIG.network_code])
                raw_count = len(getattr(raw_response, "data", []) or [])
                print("Raw SDK facility count:", raw_count)
                if raw_count:
                    reparsed = facilities_to_dataframe(raw_response)
                    print("Re-parsed facility-unit rows:", len(reparsed))
                    display(reparsed.head())
                    facilities_smoke = reparsed
                else:
                    raise RuntimeError("Raw SDK response contained zero facilities. Check API permissions or filters.")
            except Exception as exc:
                raise RuntimeError(
                    "Smoke test could not retrieve facility metadata. "
                    "If this mentions ClientConnectorError or timeout, check VPN/proxy/network access to api.openelectricity.org.au."
                ) from exc

        smoke_codes = facilities_smoke["facility_code"].dropna().drop_duplicates().head(2).tolist()
        if not smoke_codes:
            raise RuntimeError("Smoke test stopped: facility metadata did not contain usable facility codes.")
        print("Smoke-test facility codes:", smoke_codes)

        smoke_config = RetrievalConfig(
            date_start="2026-05-01T00:00:00",
            date_end="2026-05-01T01:00:00",
            facility_batch_size=2,
        )

        facility_ts_smoke = await fetch_facility_timeseries(client, smoke_codes, smoke_config)
        market_smoke = await fetch_market_data(client, smoke_config)

        if "facility_timeseries" in LAST_API_RESPONSES:
            print("Facility time-series SDK structure check:")
            debug_timeseries_structure(LAST_API_RESPONSES["facility_timeseries"])
        if "market" in LAST_API_RESPONSES:
            print("Market time-series SDK structure check:")
            debug_timeseries_structure(LAST_API_RESPONSES["market"])

    print("Requests used:", request_counter["count"])
    display(facility_ts_smoke.head())
    display(market_smoke.head())
else:
    print("Smoke test skipped. Set RUN_SMOKE_TEST = True after adding your API key.")

## 9. Full Task 1 Retrieval

After the smoke test passes, temporarily enable `RUN_FULL_RETRIEVAL`. The raw CSV files are written to `data/streaming/Task1_output/` for Task 2.


In [ ]:
# ── Task 1 Full Retrieval ────────────────────────────────────────────────────
# Run this cell AFTER the smoke test passes.
# Downloads all NEM facility metadata, 5-min power/emissions, and market data
# for the configured one-week window. Raw CSV files are saved to RAW_DIR.
# Enable: set RUN_FULL_RETRIEVAL = True in the Configuration cell (Section 1).
# NOTE: consumes approx 200-400 of your 500 daily API requests.

if RUN_FULL_RETRIEVAL:
    request_counter["count"] = 0
    verify_async_client_context_manager()

    async with get_async_oe_client() as client:
        facilities_df = await fetch_facilities(client)
        if facilities_df.empty:
            raise RuntimeError("No facilities returned; stop before requesting time-series data.")

        facility_codes = facilities_df["facility_code"].dropna().drop_duplicates().tolist()
        request_estimate = estimate_initial_requests(len(facility_codes))
        print("Request estimate:", request_estimate)

        if request_estimate["estimated_total"] > CONFIG.request_limit_per_day:
            raise RuntimeError("Estimated requests exceed the daily API limit; reduce scope or increase batch size.")

        facility_timeseries_df = await fetch_facility_timeseries(client, facility_codes)
        market_df = await fetch_market_data(client)

    facilities_path = RAW_DIR / "task1_facilities_raw.csv"
    facility_ts_path = RAW_DIR / "task1_facility_timeseries_raw.csv"
    market_path = RAW_DIR / "task1_market_raw.csv"

    facilities_df.to_csv(facilities_path, index=False)
    facility_timeseries_df.to_csv(facility_ts_path, index=False)
    market_df.to_csv(market_path, index=False)

    print("Actual API requests used:", request_counter["count"])
    print("Saved:")
    print(" -", facilities_path)
    print(" -", facility_ts_path)
    print(" -", market_path)
else:
    print("Full retrieval skipped. Set RUN_FULL_RETRIEVAL = True after the smoke test passes.")

## 10. Task 1 Validation

These checks confirm that the raw CSV files were created and contain the required metrics and time range.


In [ ]:
def validate_task1_outputs(raw_dir: Path = RAW_DIR) -> None:
    """Print validation results for the Task 1 raw CSV files."""
    facilities_path = raw_dir / "task1_facilities_raw.csv"
    facility_ts_path = raw_dir / "task1_facility_timeseries_raw.csv"
    market_path = raw_dir / "task1_market_raw.csv"

    for path in (facilities_path, facility_ts_path, market_path):
        print(path.name, "exists:", path.exists())

    if not facilities_path.exists() or not facility_ts_path.exists():
        print("Run full retrieval before validation.")
        return

    facilities = pd.read_csv(facilities_path)
    facility_ts = pd.read_csv(facility_ts_path)

    print("\nFacility metadata rows:", len(facilities))
    print("Unique facilities:", facilities["facility_code"].nunique())
    print("Rows with latitude/longitude:", facilities[["latitude", "longitude"]].dropna().shape[0])

    print("\nTime-series rows:", len(facility_ts))
    print("Metrics found:", sorted(facility_ts["metric"].dropna().astype(str).unique().tolist()))
    print("Timestamp min:", facility_ts["timestamp"].min())
    print("Timestamp max:", facility_ts["timestamp"].max())

    expected_metrics = {"power", "emissions"}
    observed_metrics = set(facility_ts["metric"].dropna().astype(str).str.lower())
    missing_metrics = expected_metrics - observed_metrics
    print("Missing core metrics:", sorted(missing_metrics) if missing_metrics else "none")

    if market_path.exists():
        market = pd.read_csv(market_path)
        print("\nMarket rows:", len(market))
        print("Market metrics:", sorted(market["metric"].dropna().astype(str).unique().tolist()))
        if "market_region" in market.columns:
            print("Market regions:", sorted(market["market_region"].dropna().astype(str).unique().tolist())[:10])


validate_task1_outputs()

## 11. Task 2: Data Cleaning, Integration, and CSV Materialisation

Task 2 transforms the raw Task 1 outputs into one facility-level analytical dataset that can be reused by later streaming and visualisation tasks. The implementation repairs noisy raw labels, restores missing regional context where possible, aggregates unit-level power/emissions to the facility level, and joins the market price/demand data needed for later MQTT and dashboard tasks.

### Why this step is necessary

- the raw facility time-series data stores labels such as `power_ADPBA1` and `emissions_ADPBA1` in the `facility_code` field
- facility-level `network_region` values are often missing in the raw cache
- market data stores labels such as `price_NSW1` and `demand_NSW1` rather than clean region codes
- non-target records such as `WEM` may still appear in the Task 1 raw outputs

### Task 2 processing logic

1. Load the three Task 1 raw CSV files.
2. Clean facility metadata and keep only operating facilities in the target NEM regions.
3. Use `unit_code` and metadata lookup tables to repair the facility time-series records.
4. Aggregate unit-level power and emissions into one row per `timestamp + facility_code`.
5. Clean and pivot market price/demand data into region-level columns.
6. Join facility-level power/emissions with market data by `timestamp + network_region`.
7. Save the cleaned outputs for Task 3 MQTT publishing and later dashboard work.

### Expected Task 2 outputs

- `Task2_output/task2_consolidated_facility_dataset.csv`
- `Task2_output/task2_facility_dimension.csv`
- `Task2_output/task2_market_clean.csv`
- `Task2_output/task2_unmatched_unit_rows.csv`


### 11.1 Task 2 processing functions

The code below is grouped into four parts: raw file loading, facility metadata cleaning, facility time-series repair/aggregation, and market-data preparation before the final merge.


In [ ]:
# Task 2 configuration and processing functions
# This block prepares the cleaned facility-level dataset used by Task 3 and later dashboard work.
import re

# Task 2 configuration

VALID_NEM_REGIONS = {'NSW1', 'QLD1', 'VIC1', 'SA1', 'TAS1'}
TASK2_OUTPUT_DIR = STREAM_DIR / 'Task2_output'
TASK2_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK2_CONSOLIDATED_PATH = TASK2_OUTPUT_DIR / 'task2_consolidated_facility_dataset.csv'
TASK2_FACILITY_DIMENSION_PATH = TASK2_OUTPUT_DIR / 'task2_facility_dimension.csv'
TASK2_MARKET_CLEAN_PATH = TASK2_OUTPUT_DIR / 'task2_market_clean.csv'
TASK2_UNMATCHED_UNITS_PATH = TASK2_OUTPUT_DIR / 'task2_unmatched_unit_rows.csv'

#### Load the three Task 1 raw CSV caches

In [ ]:
def load_task1_raw_files(raw_dir: Path = RAW_DIR) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Load the three Task 1 raw CSV files and raise FileNotFoundError if any are missing."""
    facilities_path = raw_dir / 'task1_facilities_raw.csv'
    facility_ts_path = raw_dir / 'task1_facility_timeseries_raw.csv'
    market_path = raw_dir / 'task1_market_raw.csv'

    for path in (facilities_path, facility_ts_path, market_path):
        if not path.exists():
            raise FileNotFoundError(f'Task 2 requires this Task 1 raw file: {path}')

    facilities_df = pd.read_csv(facilities_path)
    facility_ts_df = pd.read_csv(facility_ts_path)
    market_df = pd.read_csv(market_path)
    return facilities_df, facility_ts_df, market_df

#### Clean facility metadata and build a facility dimension table

In [ ]:
def collapse_fueltech(values: pd.Series) -> str | None:
    """Collapse multiple unit-level fueltech values into one facility-level label."""
    cleaned = sorted({str(value) for value in values.dropna() if str(value).strip()})
    if not cleaned:
        return None
    return cleaned[0] if len(cleaned) == 1 else 'mixed'


def prepare_facility_tables(facilities_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Clean facility metadata and build unit-level and facility-level dimension tables."""
    df = facilities_df.copy()
    df = df.dropna(subset=['facility_code', 'unit_code']).copy()
    df['network_id'] = df['network_id'].astype(str).str.strip()
    df['network_region'] = df['network_region'].astype(str).str.strip()
    df['status'] = df['status'].astype(str).str.strip().str.lower()
    df['latitude'] = pd.to_numeric(df['latitude'], errors='coerce')
    df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
    df['capacity_mw'] = pd.to_numeric(df['capacity_mw'], errors='coerce')

    df = df[df['network_id'].eq('NEM')].copy()
    df = df[df['network_region'].isin(VALID_NEM_REGIONS)].copy()
    if df['status'].notna().any():
        df = df[df['status'].eq('operating')].copy()

    unit_level_df = df.drop_duplicates(subset=['unit_code']).copy()

    facility_dim_df = (
        unit_level_df.groupby('facility_code', as_index=False)
        .agg(
            facility_name=('facility_name', 'first'),
            network_region=('network_region', 'first'),
            latitude=('latitude', 'first'),
            longitude=('longitude', 'first'),
            capacity_mw=('capacity_mw', 'sum'),
            unit_count=('unit_code', 'nunique'),
        )
    )

    fueltech_df = (
        unit_level_df.groupby('facility_code')['fueltech_id']
        .agg(collapse_fueltech)
        .reset_index(name='fueltech_id')
    )

    facility_dim_df = facility_dim_df.merge(fueltech_df, on='facility_code', how='left')
    facility_dim_df = facility_dim_df.sort_values(['network_region', 'facility_code']).reset_index(drop=True)
    return unit_level_df, facility_dim_df

#### Repair the raw facility time-series records

In [ ]:
def prepare_facility_timeseries(facility_ts_df: pd.DataFrame, unit_level_df: pd.DataFrame) -> pd.DataFrame:
    """Repair raw facility time-series labels and re-attach clean facility metadata via unit lookup."""
    df = facility_ts_df.copy()
    df = df[df['metric'].astype(str).str.lower().isin({'power', 'emissions'})].copy()
    df['metric'] = df['metric'].astype(str).str.lower()
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

    df = df.rename(columns={
        'facility_code': 'task1_facility_code_raw',
        'network_region': 'task1_network_region_raw',
    })

    # Recover the original unit code when the raw field stores labels such as power_UNITCODE.
    inferred_unit_code = df['task1_facility_code_raw'].astype(str).str.replace(r'^(power|emissions)_', '', regex=True)
    df['unit_code'] = df['unit_code'].fillna(inferred_unit_code)

    lookup_df = unit_level_df[[
        'unit_code', 'facility_code', 'facility_name', 'network_region',
        'fueltech_id', 'latitude', 'longitude'
    ]].drop_duplicates(subset=['unit_code'])

    # Reattach clean facility metadata through the unit-level lookup table.
    df = df.merge(lookup_df, on='unit_code', how='left')
    df['network_region'] = df['network_region'].fillna(df['task1_network_region_raw'])

    missing_core_mask = df[['timestamp', 'unit_code', 'metric', 'value']].isna().any(axis=1)
    orphan_mask = df[['facility_code', 'network_region']].isna().any(axis=1)

    print(f"Rows with missing core fields before cleaning: {int(missing_core_mask.sum())}")
    print(f"Rows unmatched to facility metadata: {int(orphan_mask.sum())}")

    if orphan_mask.any():
        orphan_rows = df.loc[orphan_mask].copy()
        orphan_rows.to_csv(TASK2_UNMATCHED_UNITS_PATH, index=False)
        print('Saved unmatched unit rows to:', TASK2_UNMATCHED_UNITS_PATH)
        display(
            orphan_rows[['unit_code', 'task1_facility_code_raw', 'series_name', 'task1_network_region_raw']]
            .drop_duplicates()
            .head(10)
        )
    else:
        print('No unmatched unit rows found during Task 2 repair.')

    df = df.loc[~missing_core_mask & ~orphan_mask].copy()
    non_target_region_mask = ~df['network_region'].isin(VALID_NEM_REGIONS)
    print(f"Rows outside target NEM regions: {int(non_target_region_mask.sum())}")
    df = df.loc[~non_target_region_mask].copy()
    return df.reset_index(drop=True)

#### Aggregate unit-level metrics to one row per facility and timestamp

In [ ]:
def aggregate_facility_metrics(timeseries_df: pd.DataFrame, facility_dim_df: pd.DataFrame) -> pd.DataFrame:
    """Sum unit-level power and emissions into one facility-level row per timestamp."""
    # Sum unit-level values so Task 3 receives one facility-level record per timestamp.
    grouped_df = (
        timeseries_df.groupby(['timestamp', 'facility_code', 'metric'], as_index=False)['value']
        .sum()
    )

    pivot_df = (
        grouped_df.pivot(index=['timestamp', 'facility_code'], columns='metric', values='value')
        .reset_index()
        .rename(columns={'power': 'power_mw', 'emissions': 'emissions_tco2e'})
    )

    facility_level_df = pivot_df.merge(facility_dim_df, on='facility_code', how='left')
    facility_level_df = facility_level_df.sort_values(['timestamp', 'facility_code']).reset_index(drop=True)
    return facility_level_df

#### Clean market price and demand data

In [ ]:
def prepare_market_data(market_df: pd.DataFrame) -> pd.DataFrame:
    """Clean and pivot market price/demand data into region-keyed columns."""
    df = market_df.copy()
    df = df[df['metric'].astype(str).str.lower().isin({'price', 'demand'})].copy()
    df['metric'] = df['metric'].astype(str).str.lower()
    df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce', utc=True)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')

    # Market labels may appear in different columns, so normalise them before filtering/pivoting.
    region_source = df['market_region'].fillna(df['series_name']).fillna(df['network_region'])
    df['market_region'] = region_source.astype(str).str.replace(r'^(price|demand)_', '', regex=True)
    df = df.dropna(subset=['timestamp', 'market_region', 'metric', 'value']).copy()
    df = df[df['market_region'].isin(VALID_NEM_REGIONS)].copy()

    market_clean_df = (
        df.groupby(['timestamp', 'market_region', 'metric'], as_index=False)['value']
        .mean()
        .pivot(index=['timestamp', 'market_region'], columns='metric', values='value')
        .reset_index()
        .rename(columns={'price': 'price_aud_per_mwh', 'demand': 'demand_mw'})
    )

    market_clean_df = market_clean_df.sort_values(['timestamp', 'market_region']).reset_index(drop=True)
    return market_clean_df

#### Build and save the final Task 2 outputs

In [ ]:
def build_task2_consolidated_dataset(raw_dir: Path = RAW_DIR, output_dir: Path = TASK2_OUTPUT_DIR) -> pd.DataFrame:
    """Run full Task 2 pipeline: load, clean, aggregate, join market, save to CSV, and return result."""
    facilities_raw_df, facility_ts_raw_df, market_raw_df = load_task1_raw_files(raw_dir)

    unit_level_df, facility_dim_df = prepare_facility_tables(facilities_raw_df)
    cleaned_timeseries_df = prepare_facility_timeseries(facility_ts_raw_df, unit_level_df)
    facility_level_df = aggregate_facility_metrics(cleaned_timeseries_df, facility_dim_df)
    market_clean_df = prepare_market_data(market_raw_df)

    # Join facility-level power/emissions with regional market signals.
    consolidated_df = facility_level_df.merge(
        market_clean_df,
        left_on=['timestamp', 'network_region'],
        right_on=['timestamp', 'market_region'],
        how='left',
    )

    consolidated_df = consolidated_df.drop(columns=['market_region'], errors='ignore')
    consolidated_df['timestamp_utc'] = consolidated_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    consolidated_df['timestamp_local'] = consolidated_df['timestamp'].dt.tz_convert('Australia/Sydney').dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    consolidated_df['source'] = 'Task1 raw CSV -> Task2 cleaned dataset'

    ordered_columns = [
        'timestamp_utc', 'timestamp_local', 'facility_code', 'facility_name', 'network_region',
        'fueltech_id', 'latitude', 'longitude', 'capacity_mw', 'unit_count',
        'power_mw', 'emissions_tco2e', 'price_aud_per_mwh', 'demand_mw', 'source',
    ]
    consolidated_df = consolidated_df[ordered_columns].sort_values(['timestamp_utc', 'facility_code']).reset_index(drop=True)

    output_dir.mkdir(parents=True, exist_ok=True)
    facility_dim_df.to_csv(TASK2_FACILITY_DIMENSION_PATH, index=False)
    market_clean_df.to_csv(TASK2_MARKET_CLEAN_PATH, index=False)
    consolidated_df.to_csv(TASK2_CONSOLIDATED_PATH, index=False)

    print('Task 2 outputs saved:')
    print(' -', TASK2_FACILITY_DIMENSION_PATH)
    print(' -', TASK2_MARKET_CLEAN_PATH)
    print(' -', TASK2_CONSOLIDATED_PATH)
    return consolidated_df

### 11.2 Task 2 execution and validation

This block runs the Task 2 pipeline, writes the cleaned CSV outputs, and prints summary checks such as duplicate keys, region coverage, and missing-value ratios.


In [ ]:
# Keep these switches explicit so teammates can rerun Task 2 safely when needed.
RUN_TASK2_PROCESSING = False  # set True only to rebuild CSV from Task1_output/ raw files
RUN_TASK2_VALIDATION = False


def validate_task2_output(path: Path = TASK2_CONSOLIDATED_PATH) -> None:
    """Print summary validation checks for the Task 2 consolidated CSV output."""
    if not path.exists():
        print(f'Task 2 output not found: {path}')
        print('Run Task 2 processing first.')
        return

    df = pd.read_csv(path)
    print('Task 2 output exists:', path.exists())
    print('Rows:', len(df))
    print('Columns:', list(df.columns))
    print('Duplicate timestamp_utc + facility_code rows:', int(df[['timestamp_utc', 'facility_code']].duplicated().sum()))
    print('Unique facilities:', df['facility_code'].nunique())
    print('Timestamp min:', df['timestamp_utc'].min())
    print('Timestamp max:', df['timestamp_utc'].max())
    print('Regions:', sorted(df['network_region'].dropna().astype(str).unique().tolist()))
    print('Contains only target NEM regions:', set(df['network_region'].dropna().astype(str)).issubset(VALID_NEM_REGIONS))
    print('Missing power ratio:', round(df['power_mw'].isna().mean(), 4))
    print('Missing emissions ratio:', round(df['emissions_tco2e'].isna().mean(), 4))
    if 'price_aud_per_mwh' in df.columns:
        print('Missing price ratio:', round(df['price_aud_per_mwh'].isna().mean(), 4))
    if 'demand_mw' in df.columns:
        print('Missing demand ratio:', round(df['demand_mw'].isna().mean(), 4))


# Run the Task 2 transformation pipeline and preview the cleaned output.
if RUN_TASK2_PROCESSING:
    consolidated_task2_df = build_task2_consolidated_dataset()
    display(consolidated_task2_df.head())
    display(consolidated_task2_df.tail())
else:
    print('Task 2 processing skipped. Set RUN_TASK2_PROCESSING = True to rebuild the consolidated dataset.')


# Print summary validation checks after the CSV outputs have been written.
if RUN_TASK2_VALIDATION:
    validate_task2_output()
else:
    print('Task 2 validation skipped. Set RUN_TASK2_VALIDATION = True after the consolidated CSV is created.')


## 12. Task 3: MQTT Publishing

This section implements the Task 3 publishing logic using the `paho-mqtt` Python client and the publish/subscribe architecture introduced in the streaming lecture. The publisher reads the Task 2 consolidated CSV, sorts records by event time, converts each row into a JSON-safe payload, and publishes one record at a time to an MQTT topic.

### Design choices used in this notebook

- use `timestamp_utc` as the event-time ordering field
- publish one facility-level power + emissions record per MQTT message
- wait `0.1` seconds between messages to match the assignment requirement
- store publish progress in a local state file so later tasks can reuse the same logic
- validate the workflow with a **safe sample publish** before any larger replay is considered

### Important practical note

The Task 2 consolidated CSV contains roughly **710k records**. A full historical publish at `0.1s` per message would take nearly **20 hours**, so this notebook keeps the default publish toggle in safe mode and uses a small controlled sample for validation.


### 12.1 MQTT helper functions

The helper functions below cover four responsibilities: loading the Task 2 dataset, building JSON-safe payloads, tracking publish state, and sending MQTT messages in event-time order.


In [ ]:
# Task 3 configuration and MQTT helper functions
# These helpers keep the publishing workflow modular: load data, build payloads, track state, and publish safely.
import json
import socket
import time
from datetime import timezone

try:
    import paho.mqtt.client as mqtt
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError('Install paho-mqtt first: pip install paho-mqtt') from exc

# Task 3 paths, required columns, and MQTT settings
TASK3_OUTPUT_DIR = STREAM_DIR / 'Task3_output'
TASK3_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TASK3_STATE_PATH = TASK3_OUTPUT_DIR / 'published_state.json'
TASK3_REQUIRED_COLUMNS = [
    'timestamp_utc', 'timestamp_local', 'facility_code', 'facility_name', 'network_region',
    'fueltech_id', 'latitude', 'longitude', 'capacity_mw', 'unit_count',
    'power_mw', 'emissions_tco2e', 'price_aud_per_mwh', 'demand_mw', 'source',
]
TASK3_SAMPLE_PREVIEW_COLUMNS = ['timestamp_utc', 'facility_code', 'power_mw', 'emissions_tco2e']

MQTT_HOST = os.getenv('MQTT_HOST', 'localhost')
MQTT_PORT = int(os.getenv('MQTT_PORT', '1883'))
MQTT_TOPIC = os.getenv('MQTT_TOPIC', 'comp5339/assignment2/nem/facility')
MQTT_KEEPALIVE = 60
MQTT_MESSAGE_DELAY_SECONDS = 0.1

def close_stale_duckdb_connections() -> None:
    """Close notebook-level DuckDB handles before rerunning streaming cells."""
    if 'task5_stop_subscriber' in globals():
        task5_stop_subscriber()
    elif 'task5_mqtt_client' in globals() and task5_mqtt_client is not None:
        try:
            task5_mqtt_client.disconnect()
            print('Disconnected existing Task 5 MQTT client.')
        except Exception as exc:
            print(f'Skipped Task 5 MQTT disconnect: {exc}')

    for _duckdb_conn_name in ('task5_db_conn', 'conn', '_conn_ro'):
        _duckdb_conn = globals().get(_duckdb_conn_name)
        if _duckdb_conn is not None:
            try:
                _duckdb_conn.close()
                print(f'Closed stale DuckDB connection: {_duckdb_conn_name}')
            except Exception as exc:
                print(f'Skipped stale DuckDB connection {_duckdb_conn_name}: {exc}')
            globals()[_duckdb_conn_name] = None


#### Load and validate the Task 2 dataset

In [ ]:
def load_task2_consolidated_data(path: Path = TASK2_CONSOLIDATED_PATH) -> pd.DataFrame:
    """Load the Task 2 consolidated CSV; raise FileNotFoundError if not found."""
    if not path.exists():
        raise FileNotFoundError(f'Task 3 requires the Task 2 consolidated CSV: {path}')
    return pd.read_csv(path)


def validate_task3_input_columns(df: pd.DataFrame) -> None:
    """Raise KeyError if any required Task 3 columns are missing from the DataFrame."""
    missing = [column for column in TASK3_REQUIRED_COLUMNS if column not in df.columns]
    if missing:
        raise KeyError(f'Task 3 input is missing required columns: {missing}')

#### Convert DataFrame rows into JSON-safe MQTT payloads

In [ ]:
def json_safe_value(value):
    """Convert a value to a JSON-serialisable Python type; return None for NaN/NA."""
    try:
        if pd.isna(value):
            return None
    except (TypeError, ValueError):
        pass
    if hasattr(value, 'item'):
        try:
            return value.item()
        except Exception:
            return value
    return value


def record_to_payload(record: dict) -> dict:
    """Convert a DataFrame row dict into a JSON-safe MQTT payload with a publish timestamp."""
    # Add a publisher-side timestamp so subscribers can distinguish event time from publish time.
    payload = {key: json_safe_value(value) for key, value in record.items()}
    payload['published_at_utc'] = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')
    return payload

#### Persist publish progress so later tasks can resume safely

In [ ]:
def load_publish_state(path: Path = TASK3_STATE_PATH) -> dict:
    """Load the Task 3 publish cursor from JSON; return empty state if file does not exist."""
    if not path.exists():
        return {
            'last_published_timestamp_utc': None,
            'last_published_facility_code': None,
            'records_published': 0,
        }
    return json.loads(path.read_text(encoding='utf-8'))


def save_publish_state(state: dict, path: Path = TASK3_STATE_PATH) -> None:
    """Persist the publish cursor to JSON so future runs resume from the correct position."""
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(state, indent=2), encoding='utf-8')


def reset_publish_state(path: Path = TASK3_STATE_PATH) -> None:
    """Delete the publish state file so the next run starts from the first record."""
    if path.exists():
        path.unlink()
        print('Deleted existing Task 3 publish state:', path)
    else:
        print('Task 3 publish state did not exist, so nothing was deleted.')


def prepare_task3_records(df: pd.DataFrame, state: dict | None = None, only_new_records: bool = False) -> pd.DataFrame:
    """Sort the dataset in event-time order; optionally filter to records after the last published cursor."""
    validate_task3_input_columns(df)
    working_df = df.copy()
    working_df = working_df.sort_values(['timestamp_utc', 'facility_code']).reset_index(drop=True)

    # In preview or historical replay mode, return the full event-time-sorted dataset.
    if not only_new_records:
        return working_df

    state = state or load_publish_state()
    last_timestamp = state.get('last_published_timestamp_utc')
    last_facility = state.get('last_published_facility_code')

    if not last_timestamp:
        return working_df

    filtered_df = working_df.loc[
        (working_df['timestamp_utc'] > last_timestamp)
        | (
            (working_df['timestamp_utc'] == last_timestamp)
            & (working_df['facility_code'] > str(last_facility))
        )
    ].copy()
    return filtered_df.reset_index(drop=True)

#### Preview helpers for notebook-based testing

In [ ]:
def preview_task3_messages(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """Return a DataFrame of the first n MQTT payloads for inspection without publishing."""
    preview_df = prepare_task3_records(df, only_new_records=False).head(n)
    preview_rows = [record_to_payload(row.to_dict()) for _, row in preview_df.iterrows()]
    return pd.DataFrame(preview_rows)


def publish_task3_sample(n: int = 5, reset_state_first: bool = True) -> dict:
    """Publish a small controlled sample for broker validation; resets state first by default."""
    # Resetting state makes the sample test deterministic for teammates.
    if reset_state_first:
        reset_publish_state()

    task3_source_df = load_task2_consolidated_data()
    task3_ready_df = prepare_task3_records(
        task3_source_df,
        state=load_publish_state(),
        only_new_records=False,
    ).head(n)

    if task3_ready_df.empty:
        print('No Task 3 sample records available to publish.')
        return load_publish_state()

    display(task3_ready_df[TASK3_SAMPLE_PREVIEW_COLUMNS])
    return publish_task3_records(task3_ready_df)

#### MQTT client creation and connection

In [ ]:
def on_connect(client, userdata, connect_flags, reason_code, properties):
    """MQTT on_connect callback — prints connection result code."""
    print('Connected with result code ' + str(reason_code))


def create_mqtt_client() -> mqtt.Client:
    """Create and return a paho MQTT client with VERSION2 callback API."""
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    return client


def connect_mqtt_client(client: mqtt.Client, host: str = MQTT_HOST, port: int = MQTT_PORT, keepalive: int = MQTT_KEEPALIVE) -> mqtt.Client:
    """Connect the MQTT client and start its background loop; waits 1 s for connection to establish."""
    client.connect(host, port, keepalive)
    client.loop_start()
    time.sleep(1)
    return client

#### Publish records in event-time order and update the state file

In [ ]:
def publish_task3_records(
    records_df: pd.DataFrame,
    topic: str = MQTT_TOPIC,
    delay_seconds: float = MQTT_MESSAGE_DELAY_SECONDS,
    state_path: Path = TASK3_STATE_PATH,
) -> dict:
    """Publish all records in event-time order with 0.1 s between messages; updates state file."""
    if records_df.empty:
        print('No Task 3 records to publish.')
        return load_publish_state(state_path)

    # Create one MQTT client for this publish run and reuse it for every message.
    client = create_mqtt_client()
    connect_mqtt_client(client)

    state = load_publish_state(state_path)
    published_count = 0

    try:
        for _, row in records_df.iterrows():
            payload = record_to_payload(row.to_dict())
            message = json.dumps(payload, allow_nan=False)

            # Publish synchronously so the notebook confirms each message before moving on.
            info = client.publish(topic, message, qos=1)
            info.wait_for_publish(timeout=5)

            if info.rc != mqtt.MQTT_ERR_SUCCESS:
                print(
                    f"WARNING: publish failed (rc={info.rc}) for "
                    f"{payload['facility_code']} @ {payload['timestamp_utc']} — skipping."
                )
                continue

            published_count += 1
            state['last_published_timestamp_utc'] = payload['timestamp_utc']
            state['last_published_facility_code'] = payload['facility_code']
            state['records_published'] = int(state.get('records_published', 0)) + 1
            save_publish_state(state, state_path)
            time.sleep(delay_seconds)
    finally:
        client.loop_stop()
        client.disconnect()

    print(f'Published {published_count} MQTT messages to topic: {topic}')
    print('Updated Task 3 state file:', state_path)
    return state


### 12.2 Preview and safety toggles

These toggles are designed to keep the notebook safe during development. Preview mode can stay enabled, but the full publish toggle should remain `False` unless the team intentionally wants a long historical replay.


In [ ]:
# Task 3 execution toggles and validation
# Enable preview after creating the Task 2 CSV.
RUN_TASK3_PREVIEW = False
# Keep the full publish toggle off unless the team explicitly wants a very long replay.
RUN_TASK3_PUBLISH = False  # keep False unless you intentionally want to publish the full ~710k-record dataset
RESET_TASK3_STATE = False
ONLY_NEW_RECORDS = True  # Task 6 run_continuous_execution() uses only_new_records=True internally


def validate_task3_state(path: Path = TASK3_STATE_PATH) -> None:
    """Print the current Task 3 publish state file contents for inspection."""
    if not path.exists():
        print(f'Task 3 state file does not exist yet: {path}')
        return

    state = load_publish_state(path)
    print('Task 3 state file exists:', path.exists())
    print('State contents:', state)


# Optionally delete the stored publish state before a fresh run.
if RESET_TASK3_STATE:
    reset_publish_state()
else:
    print('Task 3 state reset skipped. Set RESET_TASK3_STATE = True if you want to delete the old publish state.')


# Preview a few payloads first so the message schema can be checked before publishing.
if RUN_TASK3_PREVIEW:
    task3_source_df = load_task2_consolidated_data()
    display(preview_task3_messages(task3_source_df, n=5))
else:
    print('Task 3 preview skipped. Set RUN_TASK3_PREVIEW = True to inspect example MQTT payloads.')


# Only enable this block when the broker is ready and a long replay is intentional.
if RUN_TASK3_PUBLISH:
    task3_source_df = load_task2_consolidated_data()
    task3_state = load_publish_state()
    task3_ready_df = prepare_task3_records(
        task3_source_df,
        state=task3_state,
        only_new_records=ONLY_NEW_RECORDS,
    )
    print('Task 3 records selected for publish:', len(task3_ready_df))
    publish_task3_records(task3_ready_df)
else:
    print('Task 3 publish skipped. Set RUN_TASK3_PUBLISH = True when the MQTT broker is ready.')


# Print the current state file (if any) so the notebook records the latest publish position.
validate_task3_state()


### 12.3 Validated five-record sample publish

The following cell records the first successful end-to-end Windows test against a local Mosquitto broker. It intentionally publishes only **five records** so teammates can verify the subscriber, dashboard, or database integration without triggering a long historical replay.


In [ ]:
# Validated sample publish — intentionally sends only five records.
# Toggle RUN_TASK3_SAMPLE_PUBLISH = True to run.
RUN_TASK3_SAMPLE_PUBLISH = False

if RUN_TASK3_SAMPLE_PUBLISH:
    reset_publish_state()
    task3_source_df = load_task2_consolidated_data()
    task3_ready_df = prepare_task3_records(
        task3_source_df,
        state=load_publish_state(),
        only_new_records=False,
    ).head(5)
    display(task3_ready_df[['timestamp_utc', 'facility_code', 'power_mw', 'emissions_tco2e']])
    publish_task3_records(task3_ready_df)
else:
    print('Sample publish skipped. Set RUN_TASK3_SAMPLE_PUBLISH = True to run.')


In [ ]:
load_publish_state()

## 13. Task 3 Safe Sample Publish (Reusable)

Use this section when you want to re-run a controlled MQTT test without touching the full publish toggle. It publishes a small sample, previews the key columns, and updates `Task3_output/published_state.json`.

Recommended use cases:

- teammate testing of the MQTT subscriber
- dashboard debugging with a short message stream
- quick broker validation after moving to a new machine


In [ ]:
# Reusable helper for teammates: publish a small sample without touching the full toggle
RUN_TASK3_REUSABLE_SAMPLE = False

if RUN_TASK3_REUSABLE_SAMPLE:
    publish_task3_sample(n=5, reset_state_first=True)
else:
    print('Reusable sample publish skipped. Set RUN_TASK3_REUSABLE_SAMPLE = True to run.')


## 14. Task 4: Database Schema Implementation

Task 4 implements the schema designed in Assignment 1 into a database and extends it
with the Assignment 2 real-time streaming data.

### What this section does

1. **Copy** the Assignment 1 source DuckDB database (`ass1_energy_original.duckdb`)
   to the Task 4 output database (`energy_task4.duckdb`). The original
   `integrated_energy_wide` table (205 columns) is kept unchanged as the
   source/staging layer -- it is not modified or dropped.
2. **Inspect** the source table to confirm column names and row counts.
3. **Create 7 normalised tables** extending the Assignment 1 schema:
   - `dim_ass1_facilities` -- unified facility dimension from Assignment 1
   - `fact_nger_annual` -- NGER annual emissions per facility per year
   - `fact_abs_state_year` -- ABS state-level population per year
   - `dim_oe_facilities` -- OpenElectricity NEM facility metadata
   - `fact_realtime_generation` -- 5-minute power/emissions per facility
   - `fact_market` -- 5-minute market price/demand per NEM region
   - `bridge_facility_mapping` -- cross-reference linking OE codes to Assignment 1
4. **Populate** each table from its source.
5. **Build the bridge table** using fuzzy name matching and location proximity.
6. **Validate** all tables with row counts and integrity checks.

### Design principles

- Population operations use `INSERT OR IGNORE` where appropriate so cells can be re-run safely.
- `fact_realtime_generation` is bulk-backfilled from the Task 2 CSV; the MQTT subscriber
  upserts new rows incrementally at runtime.
- Market price/demand lives in `fact_market` only -- **not** in `fact_realtime_generation`
  (storing it there would duplicate regional values per facility row, violating 2NF).


### 14.1 Configuration

`ASS1_DB_PATH` points to the Assignment 1 DuckDB file.
All Task 4 tables are created inside this same database so both the original schema
(`integrated_energy_wide`) and the new normalised tables coexist.

Set `RUN_TASK4 = True` to execute the full pipeline.
Individual sub-section toggles let you re-run specific steps independently.


In [ ]:
import re
import math
import duckdb
import pandas as pd
from pathlib import Path
from difflib import SequenceMatcher
from IPython.display import display

# Assignment 1 source DB -- read-only, never modified
ASS1_DB_PATH = PROJECT_ROOT / 'data' / 'batch' / 'Task4_output' / 'energy_task4.duckdb'

# Task 4 output DB -- rebuilt from scratch on each full run
TASK4_OUTPUT_DIR = STREAM_DIR / 'Task4_output'
TASK4_DB_PATH    = TASK4_OUTPUT_DIR / 'energy_task4.duckdb'

# Assignment 2 Task 2 output CSVs
TASK2_OUTPUT_DIR = STREAM_DIR / 'Task2_output'

# ── Execution toggles ──────────────────────────────────────────────────────────
RUN_TASK4       = False  # master switch -- set False if DB is already built
RUN_T4_INSPECT  = True   # 14.2 inspect source DB
RUN_T4_SCHEMA   = True   # 14.3 create normalised tables
RUN_T4_ASS1     = True   # 14.4 populate Assignment 1 tables
RUN_T4_ASS2     = True   # 14.5 populate Assignment 2 tables
RUN_T4_BRIDGE   = True   # 14.6 build bridge table
RUN_T4_VALIDATE = True   # 14.7 validate schema

print('Task 4 configuration loaded.')
print('  ASS1_DB_PATH:', ASS1_DB_PATH)
print('  DB exists:', ASS1_DB_PATH.exists())
print('  TASK2_OUTPUT_DIR:', TASK2_OUTPUT_DIR)
print('  TASK4_DB_PATH:', TASK4_DB_PATH)

### 14.2 Inspect Assignment 1 Source Data

Open the database **read-only** to confirm:
- `integrated_energy_wide` exists and has the expected columns.
- The null rate on `cer_unique_key` shows how many NGER-only records exist
  (and why we cannot use it as a primary key for the facility dimension).


In [ ]:
if RUN_TASK4 and RUN_T4_INSPECT:
    # Open read-only — inspection must not modify the source table
    _conn_ro = duckdb.connect(str(ASS1_DB_PATH), read_only=True)

    # Table list
    print('Tables in Assignment 1 DB:')
    display(_conn_ro.execute('SHOW TABLES').df())

    # Confirm presence of key columns we will use in Task 4
    key_cols = [
        'facility_name', 'cer_facility_name', 'state', 'year',
        'cer_unique_key', 'cer_installed_capacity_mw', 'cer_project_status',
        'cer_accreditation_year', 'nger_fuel_category', 'cer_fuel_category',
        'nger_emissions_total', 'nger_emissions_s1', 'nger_emissions_s2',
        'nger_prod_mwh', 'nger_emission_intensity', 'nger_emission_intensity_flagged',
        'abs_estimated_resident_population_no',
        'abs_estimated_resident_population_no_erp',
        'latitude', 'longitude', 'data_source',
    ]
    schema_df = _conn_ro.execute(
        'SELECT column_name FROM information_schema.columns '
        "WHERE table_name = 'integrated_energy_wide' ORDER BY ordinal_position"
    ).df()
    present = schema_df['column_name'].tolist()
    missing = [c for c in key_cols if c not in present]
    print(f'\nSchema: {len(present)} total columns.')
    print(f'Key columns present: {len(key_cols)-len(missing)}/{len(key_cols)}')
    if missing:
        print('MISSING key columns:', missing)

    # Row count and cer_unique_key null analysis
    stats = _conn_ro.execute('''
        SELECT
            COUNT(*)                                                AS total_rows,
            COUNT(DISTINCT facility_name || chr(124) || state)     AS unique_name_state,
            SUM(CASE WHEN cer_unique_key IS NULL THEN 1 ELSE 0 END) AS nger_only_rows,
            SUM(CASE WHEN cer_unique_key IS NOT NULL THEN 1 ELSE 0 END) AS cer_matched_rows
        FROM integrated_energy_wide
    ''').df()
    print('\nSource table stats — confirms why cer_unique_key cannot be PK:')
    display(stats)

    # Sample rows
    sample = _conn_ro.execute(
        'SELECT facility_name, state, year, cer_unique_key, '
        'nger_emissions_total, latitude, longitude '
        'FROM integrated_energy_wide LIMIT 5'
    ).df()
    print('\nSample rows:')
    display(sample)

    _conn_ro.close()
    print('\nInspection complete. Source DB is intact.')
else:
    print('Inspection skipped.')


### 14.3 Helper Functions

#### `normalize_name_strict`
Re-implements the exact name-cleaning function from Assignment 1's fuzzy-matching
pipeline. Using the same normaliser ensures `source_facility_key` values are
consistent with Assignment 1's matching decisions.

#### `make_source_facility_key`
Generates the stable, collision-resistant key stored in `dim_ass1_facilities`:
- CER-matched rows  →  `CER:<cer_unique_key>`
- NGER-only rows    →  `NGER:<state>:<normalised_facility_name>`

The `CER:` / `NGER:` prefixes prevent a CER accreditation code that happens to
look like a NGER-derived string from producing a collision.

#### `haversine_km`
Great-circle distance in kilometres — used in the bridge-matching step.


In [ ]:
def _base_clean(text: str) -> str:
    """Lowercase and strip noise -- shared by both name normalisers."""
    if not text or (isinstance(text, float) and math.isnan(text)):
        return ''
    s = str(text).lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def normalize_name_strict(text: str) -> str:
    """
    Remove generic words so short OE names match verbose NGER/CER names.
    """
    if not text or (isinstance(text, float) and math.isnan(text)):
        return ''
    s = str(text).lower()
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    # Strip multi-word phrases first (order matters -- longer before shorter)
    s = re.sub(
        r'\b(hydro power station|power station|wind farm|solar farm|solar plant|'
        r'solar park|wind park|power plant|hydro power|windfarm|renewable energy|'
        r'green power hub|sugar mill|hold co|holdco|'
        r'cogeneration|cogen|csm|lfge|lfg|'
        r'facility|project|spv|trust|hub|mill|'
        r'pty|ltd|limited|australia)\b',
        ' ', s,
    )
    s = re.sub(r'\b(nsw|vic|qld|sa|wa|tas|nt|act)\b', ' ', s)
    s = re.sub(r'\b\d{4}\b', ' ', s)
    s = re.sub(r'\bstage\s*\d+\b', ' ', s)
    s = re.sub(r'\bunit\s*\d+\b', ' ', s)
    s = re.sub(r'\bhydro\b', ' ', s)
    s = re.sub(r'^the\b', '', s)
    return re.sub(r'\s+', ' ', s).strip()


def make_source_facility_key(row: dict) -> str:
    """Build a stable surrogate key for dim_ass1_facilities."""
    cer_key = row.get('cer_unique_key')
    if cer_key and str(cer_key).strip() not in ('', 'nan', 'None'):
        return f"CER:{str(cer_key).strip()}"
    state = str(row.get('state', '')).strip().upper()
    name  = normalize_name_strict(str(row.get('facility_name', '')))
    return f"NGER:{state}:{name}"


def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance in kilometres."""
    R = 6371.0
    phi1, phi2 = math.radians(float(lat1)), math.radians(float(lat2))
    dphi = math.radians(float(lat2) - float(lat1))
    dlam = math.radians(float(lon2) - float(lon1))
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2)**2
    return R * 2 * math.asin(math.sqrt(a))


print('Helper functions defined: normalize_name_strict, make_source_facility_key, haversine_km')


### 14.4 Create Normalised Schema (7 Tables)

All tables are created with `IF NOT EXISTS` — safe to re-run.
The original `integrated_energy_wide` table is **not** touched.

DuckDB supports `TIMESTAMPTZ`, `CHECK` constraints, and foreign key declarations
(FK declarations are documentary; DuckDB does not enforce them at write time by default).


In [ ]:
if RUN_TASK4 and RUN_T4_SCHEMA:
    import shutil
    if TASK4_DB_PATH.exists():
        TASK4_DB_PATH.unlink()
    TASK4_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(str(ASS1_DB_PATH), str(TASK4_DB_PATH))
    print(f"Copied Ass1 DB -> {TASK4_DB_PATH.name}")
    conn = duckdb.connect(str(TASK4_DB_PATH))

    # Table 1: dim_ass1_facilities
    # Surrogate PK because cer_unique_key is NULL for NGER-only facilities.
    conn.execute('''
        CREATE TABLE IF NOT EXISTS dim_ass1_facilities (
            ass1_facility_id     INTEGER PRIMARY KEY,
            source_facility_key  TEXT UNIQUE NOT NULL,
            source_facility_name TEXT NOT NULL,
            facility_name        TEXT NOT NULL,
            state                TEXT,
            fuel_category          TEXT,
            fuel_category_extended TEXT,
            capacity_mw            REAL,
            latitude             REAL,
            longitude            REAL,
            project_status       TEXT,
            accreditation_year   INTEGER,
            cer_unique_key       TEXT,
            data_source          TEXT
        )
    ''')

    # Table 2: fact_nger_annual
    # One row per facility per NGER reporting year.
    # Uses ass1_facility_id FK so NGER-only rows are included.
    conn.execute('''
        CREATE TABLE IF NOT EXISTS fact_nger_annual (
            ass1_facility_id           INTEGER
                REFERENCES dim_ass1_facilities(ass1_facility_id),
            year                       INTEGER NOT NULL,
            emissions_total            REAL,
            emissions_s1               REAL,
            emissions_s2               REAL,
            prod_mwh                   REAL,
            emission_intensity         REAL,
            emission_intensity_flagged INTEGER DEFAULT 0,
            data_source                TEXT,
            PRIMARY KEY (ass1_facility_id, year)
        )
    ''')

    # Table 3: fact_abs_state_year
    # ABS population at (state, year) grain -- 3NF extraction from wide table.
    _abs_cols = [
        c for c in conn.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_name = 'integrated_energy_wide'
            ORDER BY ordinal_position
        """).df()['column_name']
        if c.startswith('abs_')
    ]
    _abs_col_defs = ',\n'.join(f'        {c} REAL' for c in _abs_cols)
    conn.execute(
        'CREATE TABLE IF NOT EXISTS fact_abs_state_year (\n'
        '    state TEXT    NOT NULL,\n'
        '    year  INTEGER NOT NULL,\n'
        + _abs_col_defs + ',\n'
        '    PRIMARY KEY (state, year)\n'
        ')'
    )
    # Table 4: dim_oe_facilities
    # Assignment 2 (OpenElectricity) facility dimension.
    conn.execute('''
        CREATE TABLE IF NOT EXISTS dim_oe_facilities (
            facility_code   TEXT PRIMARY KEY,
            facility_name   TEXT,
            network_region  TEXT,
            fueltech_id     TEXT,
            latitude        REAL,
            longitude       REAL,
            capacity_mw     REAL,
            unit_count      INTEGER
        )
    ''')

    # Table 5: fact_realtime_generation
    # 5-minute facility-level power and emissions.
    # price/demand excluded — they belong in fact_market (different grain).
    conn.execute('''
        CREATE TABLE IF NOT EXISTS fact_realtime_generation (
            timestamp_utc   TIMESTAMPTZ NOT NULL,
            facility_code   TEXT NOT NULL
                REFERENCES dim_oe_facilities(facility_code),
            power_mw        REAL,
            emissions_tco2e REAL,
            PRIMARY KEY (timestamp_utc, facility_code)
        )
    ''')

    # Table 6: fact_market
    # 5-minute NEM market price and demand at region grain.
    # CHECK constraint enforces only the 5 target NEM regions.
    conn.execute('''
        CREATE TABLE IF NOT EXISTS fact_market (
            timestamp_utc     TIMESTAMPTZ NOT NULL,
            network_region    TEXT NOT NULL
                CHECK (network_region IN ('NSW1','QLD1','VIC1','SA1','TAS1')),
            price_aud_per_mwh REAL,
            demand_mw         REAL,
            PRIMARY KEY (timestamp_utc, network_region)
        )
    ''')

    # Table 7: bridge_facility_mapping
    # Cross-reference linking OE facility codes to Assignment 1 records.
    # cer_unique_key is denormalised here as a query convenience.
    conn.execute('''
        CREATE TABLE IF NOT EXISTS bridge_facility_mapping (
            oe_facility_code TEXT    NOT NULL
                REFERENCES dim_oe_facilities(facility_code),
            ass1_facility_id INTEGER NOT NULL
                REFERENCES dim_ass1_facilities(ass1_facility_id),
            cer_unique_key   TEXT,
            match_method     TEXT,
            confidence       REAL,
            PRIMARY KEY (oe_facility_code, ass1_facility_id)
        )
    ''')

    conn.close()
    print('All 7 normalised tables created (or already existed).')
else:
    print('Schema creation skipped.')


### 14.5 Populate Assignment 1 Tables

#### `dim_ass1_facilities`
The source table has one row per `(facility, year)`. We GROUP BY the facility
identifiers to collapse multi-year records into one dimension row, then generate
`source_facility_key` and assign a surrogate `ass1_facility_id`.

#### `fact_nger_annual`
Reads all `(facility, year)` rows from `integrated_energy_wide` and joins back to
`dim_ass1_facilities` to get `ass1_facility_id`. `year` is cast from DOUBLE to INTEGER.

#### `fact_abs_state_year`
Deduplicates ABS data at `(state, year)` grain using `MAX()` aggregation.


In [ ]:
if RUN_TASK4 and RUN_T4_ASS1:
    conn = duckdb.connect(str(TASK4_DB_PATH))

    # Fail fast with a clear message if the schema creation cell was skipped.
    existing_tables = set(conn.execute('SHOW TABLES').df()['name'])
    required_tables = {
        'dim_ass1_facilities',
        'fact_nger_annual',
        'fact_abs_state_year',
    }
    missing_tables = sorted(required_tables - existing_tables)
    if missing_tables:
        conn.close()
        raise RuntimeError(
            'Task 4 schema has not been created yet. '
            f'Missing tables: {missing_tables}. '
            'Run the schema creation cell before this cell.'
        )

    # ── dim_ass1_facilities ───────────────────────────────────────────────
    # Step 1: read unique facilities — collapse multi-year rows
    wide_df = conn.execute('''
        SELECT
            facility_name,
            state,
            cer_unique_key,
            MAX(cer_installed_capacity_mw)                      AS capacity_mw,
            MAX(cer_project_status)                             AS project_status,
            MAX(cer_accreditation_year)                         AS accreditation_year,
            COALESCE(MAX(nger_fuel_category), MAX(cer_fuel_category)) AS fuel_category,
            MAX(latitude)                                       AS latitude,
            MAX(longitude)                                      AS longitude,
            MAX(data_source)                                    AS data_source,
            MAX(nger_primary_fuel)                              AS nger_primary_fuel
        FROM integrated_energy_wide
        GROUP BY facility_name, state, cer_unique_key
    ''').df()

    print(f'Unique facilities in source: {len(wide_df)}')
    print(f'  CER-matched: {wide_df["cer_unique_key"].notna().sum()}')
    print(f'  NGER-only:   {wide_df["cer_unique_key"].isna().sum()}')

    # Derive fuel_category_extended: use fuel_category if available,
    # else map nger_primary_fuel to a broad category.
    _NGER_FUEL_MAP = {
        'Black Coal': 'Coal',        'Brown Coal': 'Coal',
        'Gas': 'Gas',               'Coal Seam Methane': 'Gas',
        'Waste Coal Mine Gas': 'Gas', 'Gas/Diesel': 'Gas',
        'Diesel': 'Diesel',         'Liquid Fuel': 'Diesel',
        'Kerosene': 'Diesel',       'Wind/Diesel': 'Diesel',
        'Landfill Gas': 'Biomass',  'Bagasse': 'Biomass',
        'Biogas': 'Biomass',        'Sludge Biogas': 'Biomass',
        'Biofuel': 'Biomass',       'Wood': 'Biomass',
        'Macadamia Nut Shells': 'Biomass',
        'Battery': 'Storage',
        'Wind': 'Wind', 'Solar': 'Solar', 'Hydro': 'Hydro',
    }
    wide_df['fuel_category_extended'] = wide_df['fuel_category'].where(
        wide_df['fuel_category'].notna(),
        wide_df['nger_primary_fuel'].map(_NGER_FUEL_MAP)
    )
    ext_filled = wide_df['fuel_category_extended'].notna().sum()
    print(f'  fuel_category_extended: {ext_filled} filled ({len(wide_df) - ext_filled} still NULL)')

    # Step 2: generate source_facility_key and surrogate id
    wide_df['source_facility_key']  = wide_df.apply(
        lambda r: make_source_facility_key(r.to_dict()), axis=1
    )
    wide_df['source_facility_name'] = wide_df['facility_name']
    wide_df = wide_df.reset_index(drop=True)
    wide_df['ass1_facility_id'] = wide_df.index + 1

    # Guard: detect any duplicate source_facility_key
    dup_mask = wide_df['source_facility_key'].duplicated(keep=False)
    if dup_mask.any():
        print('WARNING: duplicate source_facility_keys — appending row id to make unique')
        wide_df.loc[dup_mask, 'source_facility_key'] = (
            wide_df.loc[dup_mask, 'source_facility_key']
            + '_' + wide_df.loc[dup_mask, 'ass1_facility_id'].astype(str)
        )

    # Step 3: insert into dim_ass1_facilities
    dim_df = wide_df[[
        'ass1_facility_id', 'source_facility_key', 'source_facility_name',
        'facility_name', 'state', 'fuel_category', 'fuel_category_extended', 'capacity_mw',
        'latitude', 'longitude', 'project_status', 'accreditation_year',
        'cer_unique_key', 'data_source',
    ]].copy()
    dim_df['accreditation_year'] = pd.to_numeric(
        dim_df['accreditation_year'], errors='coerce'
    ).astype('Int64')

    conn.register('_dim_src', dim_df)
    conn.execute('''
        INSERT OR IGNORE INTO dim_ass1_facilities
        SELECT
            ass1_facility_id, source_facility_key, source_facility_name,
            facility_name, state, fuel_category, fuel_category_extended, capacity_mw,
            latitude, longitude, project_status,
            CAST(accreditation_year AS INTEGER),
            cer_unique_key, data_source
        FROM _dim_src
    ''')
    conn.unregister('_dim_src')
    n_dim = conn.execute('SELECT COUNT(*) FROM dim_ass1_facilities').fetchone()[0]
    print(f'\ndim_ass1_facilities rows: {n_dim}')

    # ── fact_nger_annual ──────────────────────────────────────────────────
    # Join source back to dim to get ass1_facility_id.
    # Match on (facility_name, state, cer_unique_key) — both NULL-safe.
    conn.execute('''
        INSERT OR IGNORE INTO fact_nger_annual
        SELECT
            d.ass1_facility_id,
            CAST(w.year AS INTEGER)                        AS year,
            w.nger_emissions_total                         AS emissions_total,
            w.nger_emissions_s1                            AS emissions_s1,
            w.nger_emissions_s2                            AS emissions_s2,
            w.nger_prod_mwh                                AS prod_mwh,
            w.nger_emission_intensity                      AS emission_intensity,
            CAST(w.nger_emission_intensity_flagged AS INTEGER) AS emission_intensity_flagged,
            w.data_source
        FROM integrated_energy_wide w
        JOIN dim_ass1_facilities d
            ON  w.facility_name = d.source_facility_name
            AND w.state         = d.state
            AND (
                   (w.cer_unique_key IS NULL AND d.cer_unique_key IS NULL)
                OR  w.cer_unique_key = d.cer_unique_key
            )
        WHERE w.year IS NOT NULL
    ''')
    n_nger = conn.execute('SELECT COUNT(*) FROM fact_nger_annual').fetchone()[0]
    print(f'fact_nger_annual rows: {n_nger}')

    # -- fact_abs_state_year ----------------------------------------
    # Deduplicate at (state, year) grain using MAX aggregation.
    _abs_cols = [
        c for c in conn.execute("""
            SELECT column_name FROM information_schema.columns
            WHERE table_name = 'integrated_energy_wide'
            ORDER BY ordinal_position
        """).df()['column_name']
        if c.startswith('abs_')
    ]
    _abs_select = ',\n'.join(
        f'            MAX({c}) AS {c}' for c in _abs_cols
    )
    conn.execute(
        'INSERT OR IGNORE INTO fact_abs_state_year\n'
        'SELECT\n'
        '    state,\n'
        '    CAST(year AS INTEGER) AS year,\n'
        + _abs_select + '\n'
        'FROM integrated_energy_wide\n'
        'WHERE state IS NOT NULL AND year IS NOT NULL\n'
        'GROUP BY state, CAST(year AS INTEGER)'
    )
    n_abs = conn.execute('SELECT COUNT(*) FROM fact_abs_state_year').fetchone()[0]
    print(f'fact_abs_state_year rows: {n_abs}')

    conn.close()
    print('\nAssignment 1 tables populated successfully.')
else:
    print('Assignment 1 table population skipped.')


### 14.6 Populate Assignment 2 Tables

#### `dim_oe_facilities`
Loaded from `task2_facility_dimension.csv` — one row per OpenElectricity facility.

#### `fact_realtime_generation` (bulk backfill)
Loaded from `task2_consolidated_facility_dataset.csv`.
**Only 4 columns are loaded**: `timestamp_utc`, `facility_code`, `power_mw`, `emissions_tco2e`.
`price_aud_per_mwh` and `demand_mw` are present in the CSV (from the Task 2 market join)
but are **deliberately excluded** — they belong in `fact_market` to avoid 2NF violation.
The MQTT subscriber will upsert new rows at runtime; the PRIMARY KEY makes it idempotent.

#### `fact_market`
Loaded from `task2_market_clean.csv`.
Column renames on import: `timestamp` → `timestamp_utc`, `market_region` → `network_region`.
Pre-filtered to valid NEM regions before INSERT to satisfy the CHECK constraint.


In [ ]:
if RUN_TASK4 and RUN_T4_ASS2:
    conn = duckdb.connect(str(TASK4_DB_PATH))

    # ── dim_oe_facilities ─────────────────────────────────────────────────
    oe_fac_path = TASK2_OUTPUT_DIR / 'task2_facility_dimension.csv'
    if not oe_fac_path.exists():
        print(f'WARNING: {oe_fac_path} not found — dim_oe_facilities not loaded.')
    else:
        oe_fac = pd.read_csv(oe_fac_path)
        for col in ('capacity_mw', 'latitude', 'longitude'):
            oe_fac[col] = pd.to_numeric(oe_fac[col], errors='coerce')
        oe_fac['unit_count'] = pd.to_numeric(oe_fac['unit_count'], errors='coerce')

        conn.register('_oe_fac', oe_fac)
        conn.execute('''
            INSERT OR IGNORE INTO dim_oe_facilities
            SELECT facility_code, facility_name, network_region, fueltech_id,
                   latitude, longitude, capacity_mw, CAST(unit_count AS INTEGER)
            FROM _oe_fac
        ''')
        conn.unregister('_oe_fac')
        n = conn.execute('SELECT COUNT(*) FROM dim_oe_facilities').fetchone()[0]
        print(f'dim_oe_facilities rows: {n}')

    # ── fact_realtime_generation (bulk backfill) ──────────────────────────
    ts_path = TASK2_OUTPUT_DIR / 'task2_consolidated_facility_dataset.csv'
    if not ts_path.exists():
        print(f'WARNING: {ts_path} not found — fact_realtime_generation not backfilled.')
    else:
        print('Loading consolidated CSV for bulk backfill (may take a moment)...')
        # Load ONLY the 4 columns needed — price/demand deliberately excluded
        ts_df = pd.read_csv(
            ts_path,
            usecols=['timestamp_utc', 'facility_code', 'power_mw', 'emissions_tco2e'],
        )
        ts_df['timestamp_utc']  = pd.to_datetime(ts_df['timestamp_utc'], utc=True)
        ts_df['power_mw']       = pd.to_numeric(ts_df['power_mw'],        errors='coerce')
        ts_df['emissions_tco2e']= pd.to_numeric(ts_df['emissions_tco2e'], errors='coerce')

        print(f'  Loaded {len(ts_df):,} rows — inserting into fact_realtime_generation...')
        conn.register('_ts', ts_df)
        conn.execute('''
            INSERT OR IGNORE INTO fact_realtime_generation
            SELECT timestamp_utc::TIMESTAMPTZ, facility_code, power_mw, emissions_tco2e
            FROM _ts
        ''')
        conn.unregister('_ts')
        n = conn.execute('SELECT COUNT(*) FROM fact_realtime_generation').fetchone()[0]
        print(f'  fact_realtime_generation rows: {n:,}')

    # ── fact_market ───────────────────────────────────────────────────────
    mkt_path = TASK2_OUTPUT_DIR / 'task2_market_clean.csv'
    if not mkt_path.exists():
        print(f'WARNING: {mkt_path} not found — fact_market not loaded.')
    else:
        mkt_df = pd.read_csv(mkt_path)
        # Rename to match schema column names
        mkt_df = mkt_df.rename(columns={
            'timestamp':     'timestamp_utc',
            'market_region': 'network_region',
        })
        mkt_df['timestamp_utc']     = pd.to_datetime(mkt_df['timestamp_utc'], utc=True)
        mkt_df['price_aud_per_mwh'] = pd.to_numeric(mkt_df['price_aud_per_mwh'], errors='coerce')
        mkt_df['demand_mw']         = pd.to_numeric(mkt_df['demand_mw'],         errors='coerce')
        # Pre-filter to valid NEM regions to satisfy the CHECK constraint
        valid_regions = {'NSW1', 'QLD1', 'VIC1', 'SA1', 'TAS1'}
        mkt_df = mkt_df[mkt_df['network_region'].isin(valid_regions)].copy()

        conn.register('_mkt', mkt_df)
        conn.execute('''
            INSERT OR IGNORE INTO fact_market
            SELECT timestamp_utc::TIMESTAMPTZ, network_region,
                   price_aud_per_mwh, demand_mw
            FROM _mkt
        ''')
        conn.unregister('_mkt')
        n = conn.execute('SELECT COUNT(*) FROM fact_market').fetchone()[0]
        print(f'fact_market rows: {n:,}')

    conn.close()
    print('\nAssignment 2 tables populated successfully.')
else:
    print('Assignment 2 table population skipped.')


### 14.7 Build Bridge Table (`bridge_facility_mapping`)

#### Matching strategy

1. **Fuel-type pre-filter**: All OE facility types (renewable *and* non-renewable:
   wind, solar, hydro, biomass, coal, gas, distillate, pumps, mixed) are candidates.
   OE fueltech types not listed in `FUEL_COMPAT` (e.g. unknown battery-only types)
   are excluded from automatic matching and collected for manual review.
2. **State alignment**: Map `network_region` (e.g. `NSW1`) to Australian state
   (e.g. `NSW`) and restrict candidates to the same state.
3. **Fuel-type compatibility filter**: OE facility types are only matched against
   Assignment 1 facilities with a compatible `fuel_category_extended` to prevent
   cross-type false positives (e.g. wind matched to coal).
   Unknown OE types (not in `FUEL_COMPAT`) return `False` and require manual review.
   `mixed` is listed in `FUEL_COMPAT` but gated by an additional name/distance check.
4. **BESS / battery protection**: OE facilities whose name contains `bess` or
   `battery`, or whose facility code ends in `bes` (e.g. `CAPBES`, `BRNDBES`),
   are not auto-matched to A1 facilities that lack those keywords,
   preventing co-located BESS from inheriting a wind/solar facility's record.
5. **Name similarity (primary method)**: `difflib.SequenceMatcher` on normalised
   names after stripping generic suffixes. Threshold: similarity >= 0.75.
6. **Location proximity (supporting boost)**: When both records have valid
   coordinates within 10 km, a location score boosts the combined score.
   Location evidence can support but never penalise a strong name match.
7. **Best match per OE facility**: Only the highest-scoring compatible Assignment 1
   candidate is inserted. Facilities below threshold remain unmatched.

#### Notes on one-to-many matches

Some Assignment 1 facilities are matched by multiple OE facility codes
(e.g. `LIMOSF1` + `LIMOSF2` both map to *Limondale Solar Farm*). This is
expected: OpenElectricity can represent physical sub-facilities or unit stages
with separate codes, while Assignment 1's CER/NGER data records the whole
station as one entry.

#### Manual review output

After automatic matching, a summary of OE facilities requiring manual review
is printed below the bridge table statistics. These are BESS/battery-named
facilities (intentionally excluded) and any `mixed`-fueltech facilities that
did not meet the stricter name/distance thresholds for automatic matching.

#### Coverage interpretation

The bridge table keeps only high-confidence, fuel-compatible matches. Some OE
facilities remain unmatched because Assignment 1 and OpenElectricity differ in
coverage, naming, and facility granularity. BESS OE facilities are always flagged for manual review. `mixed`-type facilities
are auto-matched only when name similarity >= 0.90 or name >= 0.75 with location
within 10 km; otherwise they are flagged for manual review.


In [ ]:
if RUN_TASK4 and RUN_T4_BRIDGE:
    conn = duckdb.connect(str(TASK4_DB_PATH))

    # Always clear before rebuilding — INSERT OR IGNORE keeps stale rows
    conn.execute('DELETE FROM bridge_facility_mapping')
    print('Cleared existing bridge rows.')

    oe_df   = conn.execute('SELECT * FROM dim_oe_facilities').df()
    ass1_df = conn.execute(
        'SELECT ass1_facility_id, source_facility_name, state, '
        'cer_unique_key, latitude, longitude, fuel_category, fuel_category_extended FROM dim_ass1_facilities'
    ).df()

    region_to_state   = {'NSW1':'NSW','QLD1':'QLD','VIC1':'VIC','SA1':'SA','TAS1':'TAS'}
    # Match all OE facility types including coal and gas
    renewable_tokens  = {'wind','solar','hydro','biomass','landfill','wcmg','biogas',
                         'coal','gas','distillate','recip','ccgt','ocgt','steam','mixed','pumps'}
    THRESHOLD         = 0.75
    LOCATION_RADIUS_KM = 10.0

    # OE fueltech -> compatible A1 fuel_category (None = A1 fuel unknown, accept any)
    FUEL_COMPAT = {
        # Renewables
        'wind':              {'wind'},
        'solar_utility':     {'solar'},
        'solar_rooftop':     {'solar'},
        'hydro':             {'hydro'},
        'gas_wcmg':          {'biomass', 'gas'},
        'bioenergy_biomass': {'biomass'},
        'bioenergy_biogas':  {'biomass'},
        'landfill_gas':      {'biomass'},
        # Non-renewables
        'coal_black':        {'coal'},
        'coal_brown':        {'coal'},
        'gas_ocgt':          {'gas'},
        'gas_ccgt':          {'gas'},
        'gas_recip':         {'gas'},
        'gas_steam':         {'gas'},
        'distillate':        {'diesel'},
        'pumps':             {'hydro', 'storage'},
        # Mixed: matched only under stricter name/location safeguards (see loop below);
        # broad fuel set kept so fuel_types_compatible passes -- the real gate is
        # the extra combined/distance check applied when fueltech == 'mixed'.
        'mixed':             {'wind', 'solar', 'hydro', 'biomass', 'coal', 'gas', 'diesel', 'storage'},
    }

    def _is_bess(name, code=''):
        n = str(name).lower()
        c = str(code).lower()
        # 'bes' suffix covers CAPBES / BRNDBES codes (no double-s)
        return ('bess' in n or 'battery' in n or
                'bess' in c or 'batt' in c or c.endswith('bes'))

    def fuel_types_compatible(oe_ft, a1_fc):
        """Return True if OE fueltech and Assignment 1 fuel category are compatible for bridge matching."""
        oe_key = str(oe_ft).lower().strip()
        # Use fuel_category_extended for broader matching including coal/gas
        a1_fc_ext = a1.get('fuel_category_extended') or a1_fc
        a1_val = None if (not a1_fc_ext or str(a1_fc_ext).lower() in ('nan','none','')) \
                 else str(a1_fc_ext).lower().strip()
        allowed = FUEL_COMPAT.get(oe_key)
        if allowed is None:
            return False  # unknown OE type: requires manual review
        # Bug fix: None in allowed was never True (FUEL_COMPAT has no None entries).
        # A1 fuel unknown (NULL) should pass any OE fueltech -- name/location are the gate.
        if a1_val is None:
            return True
        return a1_val in allowed

    matches = []
    for _, oe in oe_df.iterrows():
        fueltech = str(oe.get('fueltech_id', '')).lower()
        if not any(tok in fueltech for tok in renewable_tokens):
            continue
        oe_state = region_to_state.get(str(oe.get('network_region', '')))
        if not oe_state:
            continue
        candidates = ass1_df[ass1_df['state'] == oe_state].sort_values('source_facility_name')
        if candidates.empty:
            continue
        oe_name_n = normalize_name_strict(str(oe['facility_name']))
        oe_lat = oe.get('latitude'); oe_lon = oe.get('longitude')
        has_oe = oe_lat is not None and pd.notna(oe_lat) and float(oe_lat) != 0
        best_score=0.0; best_match=None; best_method=None
        for _, a1 in candidates.iterrows():
            a1_name_n  = normalize_name_strict(str(a1['source_facility_name']))
            name_score = SequenceMatcher(None, oe_name_n, a1_name_n).ratio()
            a1_lat=a1.get('latitude'); a1_lon=a1.get('longitude')
            has_a1 = a1_lat is not None and pd.notna(a1_lat) and float(a1_lat)!=0
            if has_oe and has_a1:
                try:
                    dist = haversine_km(oe_lat, oe_lon, a1_lat, a1_lon)
                    if dist < LOCATION_RADIUS_KM:
                        location_score = max(0.0, 1 - dist / LOCATION_RADIUS_KM)
                        # Location is supporting evidence only: it should not penalise
                        # an otherwise strong name/fuel/state match.
                        combined = max(name_score, 0.7 * name_score + 0.3 * location_score)
                        method = 'location_proximity'
                    else:
                        combined = name_score
                        method = 'name_fuzzy'
                except Exception:
                    combined=name_score; method='name_fuzzy'
            else:
                combined=name_score; method='name_fuzzy'
            # Only update best if this candidate is fuel-compatible (or fuel unknown)
            # This prevents a high-scoring but incompatible candidate from blocking
            # a compatible one with the same score (e.g. Wind Farm vs Solar Farm
            # both normalising to the same short name like "white rock").
            # Block BESS OE from matching non-BESS A1 facilities
            if _is_bess(oe['facility_name'], oe.get('facility_code','')) and not _is_bess(a1['source_facility_name']):
                continue
            compat = fuel_types_compatible(oe.get('fueltech_id',''), a1.get('fuel_category_extended') or a1.get('fuel_category'))
            # mixed fueltech: require very high name similarity OR name+proximity
            if fueltech == 'mixed':
                within_radius = (method == 'location_proximity')
                if combined < 0.90 and not (combined >= THRESHOLD and within_radius):
                    continue
            if compat and combined > best_score:
                best_score=combined; best_match=a1; best_method=method

        fuel_ok = best_match is not None
        if best_score >= THRESHOLD and fuel_ok:
            matches.append({
                'oe_facility_code': oe['facility_code'],
                'ass1_facility_id': int(best_match['ass1_facility_id']),
                'cer_unique_key':   best_match.get('cer_unique_key'),
                'match_method':     best_method,
                'confidence':       round(best_score, 4),
            })

    bridge_df = pd.DataFrame(matches)
    print(f'Matches (confidence >= {THRESHOLD}, fuel-compatible): {len(bridge_df)}')
    if not bridge_df.empty:
        conn.register('_b', bridge_df)
        conn.execute('''
            INSERT OR IGNORE INTO bridge_facility_mapping
            SELECT oe_facility_code, CAST(ass1_facility_id AS INTEGER),
                   cer_unique_key, match_method, confidence FROM _b
        ''')
        conn.unregister('_b')
    min_conf = conn.execute('SELECT MIN(confidence) FROM bridge_facility_mapping').fetchone()[0]
    n = conn.execute('SELECT COUNT(*) FROM bridge_facility_mapping').fetchone()[0]
    print(f'bridge_facility_mapping: {n} rows | MIN confidence: {min_conf:.4f}')
    assert min_conf >= THRESHOLD, f'FAIL: MIN confidence {min_conf:.4f} < {THRESHOLD}'
    print('PASS: all matches meet confidence threshold and fuel-type compatibility.')
    display(bridge_df.sort_values('confidence', ascending=False).head(10))
    # Manual review: unmatched OE with mixed fueltech or BESS name
    matched_codes = set(bridge_df['oe_facility_code']) if not bridge_df.empty else set()
    manual = []
    for _, oe_r in oe_df.iterrows():
        if oe_r['facility_code'] in matched_codes:
            continue
        ft = str(oe_r.get('fueltech_id', '')).lower()
        nm = str(oe_r.get('facility_name', ''))
        oe_state = region_to_state.get(str(oe_r.get('network_region', '')))
        if not oe_state:
            continue
        code_r = str(oe_r.get('facility_code', ''))
        if _is_bess(nm, code_r):
            manual.append({'oe_code': code_r, 'oe_name': nm,
                           'oe_fuel': ft, 'state': oe_state,
                           'reason': 'BESS/battery — verify co-location'})
        elif ft not in FUEL_COMPAT and ft not in ('', 'nan', 'none'):
            manual.append({'oe_code': code_r, 'oe_name': nm,
                           'oe_fuel': ft, 'state': oe_state,
                           'reason': f'unknown fueltech: {ft}'})
    if manual:
        import pandas as _pd
        print(f'\nManual review needed ({len(manual)} unmatched facilities):')
        display(_pd.DataFrame(manual)[['oe_code','oe_name','oe_fuel','state','reason']])
    else:
        print('No manual review candidates.')
    conn.close()
else:
    print('Bridge table build skipped.')


### 14.10 Manual Bridge Overrides

A small whitelist of high-confidence matches that the automatic algorithm cannot reach due to missing A1 coordinates (BESS co-location), compound OE facility codes (location-only), or A1 fuel data entry errors.

In [ ]:
# ── Task 4 Manual Bridge Overrides ────────────────────────────────────────────
# Adds high-confidence matches that the automatic algorithm cannot reach:
#   - BESS co-location: name matches perfectly but A1 coordinates are missing
#   - Location-only: distance=0 km but names differ (multi-asset OE code)
#   - Manual review: strong name+location but A1 fuel field appears incorrect
#
# Each tuple: (oe_facility_code, a1_source_facility_name, match_method)
MANUAL_BRIDGE_OVERRIDES = [
    # BESS units co-located with the named A1 facility (A1 coords missing -> auto-match failed)
    ('WDBESS',   'WESTERN DOWNS GREEN POWER HUB PTY LTD', 'bess_colocation'),
    ('WDBESS2',  'WESTERN DOWNS GREEN POWER HUB PTY LTD', 'bess_colocation'),
    ('BALBESS',  'Ballarat Renewable Energy Facility',    'bess_colocation'),
    ('LIMBESS',  'Limondale Solar Farm',                  'bess_colocation'),
    # Location-only: OE uses a compound multi-asset code; distance to A1 record = 0.00 km
    ('LI_WY_CA', 'Liapootah Hydro Power Station',         'location_only'),
    # Manual review: name_score=0.92, distance=0.02 km; A1 fuel_category='Solar'
    # appears to be a data-entry error for this coal_black facility.
    ('VP',       'Vales Point Power Station',             'manual_review'),
    # Near-miss: name clearly refers to same facility; A1 coords appear offset
    ('GANNBESS', 'Gannawarra Solar Farm Pty Ltd',         'bess_colocation'),
    # VALDORA removed: OE=Valdora solar farm, A1=Sunshine Coast Airport solar;
    # same region but different installations (6.95 km, different sites)
    # Near-miss: OE short name vs A1 full name; 10 km (dam vs powerhouse)
    ('RUBICON',   'Rubicon Hydro Scheme',                        'name_location'),
    # BESS at coal plant, blocked by _is_bess() check (A1 fuel ext=Coal)
    ('LDBESS',    'Liddell Power Station',                       'bess_colocation'),
    ('HBESS',     'Hazelwood Power Station',                     'bess_colocation'),
    # Wind farm: mixed threshold 0.90, name_score=0.857 just below; same facility by name
    ('GULLRGWF',  'New Gullen Range Wind Farm Pty Ltd',          'name_location'),
    # Distillate unit at same site; fuel compatible (distillate->diesel); name=0.73
    ('STANVAC',   'Lonsdale/Port Stanvac Power Station',         'name_location'),
    # Gas turbine, 0.01 km; combined score 0.737 just under 0.75 threshold
    ('SITHE',     'Smithfield Gas Turbine',                      'name_location'),
    # Hydro multi-asset OE code; 5.9 km, partial name match
    ('MCKAY',     'Bogong Power Station',                        'name_location'),
    # BESS in A1 too; 22 km gap likely A1 coord error; name=0.80
    ('CHBESS',    'Chinchilla BESS',                             'bess_colocation'),
    # BESS at gas plant, 0.19 km; mixed threshold blocks auto-match
    ('SMTHBES1',  'Smithfield Gas Turbine',                      'bess_colocation'),
    # Distillate auxiliary unit at Eraring coal plant; name=1.0, 0.19 km;
    # fuel_category_extended='Coal' blocks auto-match (distillate not in Coal compat set)
    ('ERGT01',    'Eraring Power Station',                       'name_location'),
]

if RUN_TASK4:
    conn = duckdb.connect(str(TASK4_DB_PATH))
    inserted, skipped = 0, []

    for oe_code, a1_name, method in MANUAL_BRIDGE_OVERRIDES:
        # Skip if already matched by the automatic pass
        already = conn.execute(
            'SELECT 1 FROM bridge_facility_mapping WHERE oe_facility_code = ?',
            [oe_code]
        ).fetchone()
        if already:
            print(f'  SKIP (already matched): {oe_code}')
            continue

        row = conn.execute(
            'SELECT ass1_facility_id FROM dim_ass1_facilities WHERE source_facility_name = ?',
            [a1_name]
        ).fetchone()
        if row is None:
            skipped.append((oe_code, a1_name))
            print(f'  NOT FOUND in A1: {a1_name} <- check spelling')
            continue

        conn.execute(
            '''INSERT OR IGNORE INTO bridge_facility_mapping
               (oe_facility_code, ass1_facility_id, cer_unique_key, match_method, confidence)
               VALUES (?, ?, NULL, ?, 0.95)''',
            [oe_code, row[0], method]
        )
        inserted += 1
        print(f'  OK  {oe_code:12s} -> {a1_name} [{method}]')

    conn.commit()
    n_total = conn.execute('SELECT COUNT(*) FROM bridge_facility_mapping').fetchone()[0]
    conn.close()
    print(f'Manual overrides inserted : {inserted}')
    print(f'Skipped (name not found)  : {len(skipped)}')
    print(f'bridge_facility_mapping total: {n_total} rows')
else:
    print('Manual bridge overrides skipped (RUN_TASK4 = False).')


### 14.8 Diagnose Unmatched Bridge Candidates

This diagnostic step does **not** modify `bridge_facility_mapping`. It reviews renewable
OpenElectricity facilities that remain unmatched and records the best Assignment 1
candidate by region/state, coordinates, fuel compatibility, name similarity, and location proximity.

The goal is to identify possible near-misses for manual review while keeping the bridge
table conservative and high-confidence.


In [ ]:
if RUN_TASK4 and RUN_T4_BRIDGE:
    conn = duckdb.connect(str(TASK4_DB_PATH), read_only=True)

    oe_unmatched = conn.execute('''
        SELECT oe.*
        FROM dim_oe_facilities oe
        LEFT JOIN bridge_facility_mapping b
          ON oe.facility_code = b.oe_facility_code
        WHERE b.oe_facility_code IS NULL
    ''').df()
    ass1_df = conn.execute('''
        SELECT ass1_facility_id, source_facility_name, state,
               cer_unique_key, latitude, longitude, fuel_category, fuel_category_extended
        FROM dim_ass1_facilities
    ''').df()
    conn.close()

    region_to_state = {'NSW1':'NSW', 'QLD1':'QLD', 'VIC1':'VIC', 'SA1':'SA', 'TAS1':'TAS'}
    renewable_tokens = {'wind', 'solar', 'hydro', 'biomass', 'landfill', 'wcmg', 'biogas',
                    'coal', 'gas', 'distillate', 'recip', 'ccgt', 'ocgt', 'steam', 'mixed', 'pumps'}
    THRESHOLD = 0.75
    LOCATION_RADIUS_KM = 10.0
    FUEL_COMPAT = {
        # Renewables
        'wind':              {'wind'},
        'solar_utility':     {'solar'},
        'solar_rooftop':     {'solar'},
        'hydro':             {'hydro'},
        'gas_wcmg':          {'biomass', 'gas'},
        'bioenergy_biomass': {'biomass'},
        'bioenergy_biogas':  {'biomass'},
        'landfill_gas':      {'biomass'},
        # Non-renewables
        'coal_black':        {'coal'},
        'coal_brown':        {'coal'},
        'gas_ocgt':          {'gas'},
        'gas_ccgt':          {'gas'},
        'gas_recip':         {'gas'},
        'gas_steam':         {'gas'},
        'distillate':        {'diesel'},
        'pumps':             {'hydro', 'storage'},
        # Mixed: matched only under stricter name/location safeguards
        'mixed':             {'wind', 'solar', 'hydro', 'biomass', 'coal', 'gas', 'diesel', 'storage'},
    }

    def _fuel_types_compatible(oe_ft, a1_fc):
        oe_key = str(oe_ft).lower().strip()
        a1_val = None if (not a1_fc or str(a1_fc).lower() in ('nan', 'none', '')) \
                 else str(a1_fc).lower().strip()
        allowed = FUEL_COMPAT.get(oe_key)
        if allowed is None:
            return True
        return (a1_val is None and None in allowed) or (a1_val in allowed)

    def _score_candidate(oe, a1):
        oe_name_n = normalize_name_strict(str(oe['facility_name']))
        a1_name_n = normalize_name_strict(str(a1['source_facility_name']))
        name_score = SequenceMatcher(None, oe_name_n, a1_name_n).ratio()
        dist_km = None
        method = 'name_fuzzy'

        oe_lat, oe_lon = oe.get('latitude'), oe.get('longitude')
        a1_lat, a1_lon = a1.get('latitude'), a1.get('longitude')
        has_oe = oe_lat is not None and pd.notna(oe_lat) and float(oe_lat) != 0
        has_a1 = a1_lat is not None and pd.notna(a1_lat) and float(a1_lat) != 0
        combined = name_score
        if has_oe and has_a1:
            try:
                dist_km = haversine_km(oe_lat, oe_lon, a1_lat, a1_lon)
                if dist_km < LOCATION_RADIUS_KM:
                    location_score = max(0.0, 1 - dist_km / LOCATION_RADIUS_KM)
                    # Keep diagnostics consistent with the bridge builder:
                    # location can boost/support, but never lower name similarity.
                    combined = max(name_score, 0.7 * name_score + 0.3 * location_score)
                    method = 'location_proximity'
            except Exception:
                pass
        return name_score, combined, dist_km, method

    rows = []
    for _, oe in oe_unmatched.iterrows():
        fueltech = str(oe.get('fueltech_id', '')).lower()
        if not any(tok in fueltech for tok in renewable_tokens):
            continue

        oe_state = region_to_state.get(str(oe.get('network_region', '')))
        candidates = ass1_df[ass1_df['state'] == oe_state].copy() if oe_state else ass1_df.iloc[0:0].copy()

        best_any = None
        best_compat = None
        for _, a1 in candidates.iterrows():
            name_score, combined, dist_km, method = _score_candidate(oe, a1)
            compat = _fuel_types_compatible(oe.get('fueltech_id', ''), a1.get('fuel_category'))
            item = {
                'ass1_facility_id': a1['ass1_facility_id'],
                'ass1_name': a1['source_facility_name'],
                'ass1_state': a1.get('state'),
                'ass1_fuel_category': a1.get('fuel_category'),
                'ass1_latitude': a1.get('latitude'),
                'ass1_longitude': a1.get('longitude'),
                'name_score': round(name_score, 4),
                'combined_score': round(combined, 4),
                'distance_km': None if dist_km is None else round(dist_km, 2),
                'method': method,
                'fuel_compatible': compat,
            }
            if best_any is None or item['combined_score'] > best_any['combined_score']:
                best_any = item
            # Mirror the bridge builder's mixed strict rule so diagnostic reflects real eligibility
            passes_mixed = True
            if fueltech == 'mixed':
                within_radius = (method == 'location_proximity')
                if combined < 0.90 and not (combined >= THRESHOLD and within_radius):
                    passes_mixed = False
            if compat and passes_mixed and (best_compat is None or item['combined_score'] > best_compat['combined_score']):
                best_compat = item

        if not oe_state:
            reason = 'no_state_mapping'
        elif candidates.empty:
            reason = 'no_same_state_candidates'
        elif best_compat is None:
            reason = 'no_fuel_compatible_candidate'
        elif best_compat['combined_score'] >= 0.65:
            reason = 'near_miss_below_threshold'
        else:
            reason = 'low_similarity'

        best = best_compat or best_any or {}
        rows.append({
            'oe_facility_code': oe.get('facility_code'),
            'oe_facility_name': oe.get('facility_name'),
            'oe_network_region': oe.get('network_region'),
            'oe_state': oe_state,
            'oe_fueltech_id': oe.get('fueltech_id'),
            'oe_latitude': oe.get('latitude'),
            'oe_longitude': oe.get('longitude'),
            'reason': reason,
            'best_ass1_facility_id': best.get('ass1_facility_id'),
            'best_ass1_name': best.get('ass1_name'),
            'ass1_state': best.get('ass1_state'),
            'best_ass1_fuel_category': best.get('ass1_fuel_category'),
            'ass1_latitude': best.get('ass1_latitude'),
            'ass1_longitude': best.get('ass1_longitude'),
            'name_score': best.get('name_score'),
            'combined_score': best.get('combined_score'),
            'distance_km': best.get('distance_km'),
            'match_method': best.get('method'),
            'fuel_compatible': best.get('fuel_compatible'),
            'best_any_name': None if best_any is None else best_any.get('ass1_name'),
            'best_any_score': None if best_any is None else best_any.get('combined_score'),
            'best_any_fuel_compatible': None if best_any is None else best_any.get('fuel_compatible'),
        })

    diag_df = pd.DataFrame(rows).sort_values(
        ['combined_score', 'name_score'], ascending=False, na_position='last'
    )
    diag_path = TASK4_OUTPUT_DIR / 'task4_unmatched_bridge_diagnostics.csv'
    diag_df.to_csv(diag_path, index=False)

    print(f'Unmatched renewable OE facilities diagnosed: {len(diag_df)}')
    print(f'Diagnostic CSV written to: {diag_path}')
    print('\nReason summary:')
    display(diag_df['reason'].value_counts(dropna=False).rename_axis('reason').reset_index(name='count'))

    print('\nTop possible near-misses for manual review:')
    review_cols = [
        'oe_facility_code', 'oe_facility_name', 'oe_network_region', 'oe_state',
        'oe_fueltech_id', 'oe_latitude', 'oe_longitude', 'reason',
        'best_ass1_name', 'ass1_state', 'best_ass1_fuel_category',
        'ass1_latitude', 'ass1_longitude', 'combined_score', 'name_score',
        'distance_km', 'fuel_compatible',
    ]
    display(diag_df[review_cols].head(20))
else:
    print('Unmatched bridge diagnostics skipped.')


### 14.9 Schema Validation

1. **Row counts** for all 8 tables (source + 7 normalised).
2. **Primary key uniqueness** -- zero duplicate PKs expected.
3. **FK NULL rates** -- critical FK columns should be 0% null.
4. **Orphan FK checks** -- every fact/bridge FK should resolve to a dimension row.
5. **Bridge coverage rate** -- what % of OpenElectricity facilities matched.
6. **Sample cross-dataset query** -- joins real-time generation, historical NGER,
   and market data for matched facilities.


In [ ]:
if RUN_TASK4 and RUN_T4_VALIDATE:
    conn = duckdb.connect(str(TASK4_DB_PATH), read_only=True)

    # 1. Row counts
    print('=== Row counts ===')
    all_tables = [
        'integrated_energy_wide',   # source/staging layer — unchanged
        'dim_ass1_facilities',
        'fact_nger_annual',
        'fact_abs_state_year',
        'dim_oe_facilities',
        'fact_realtime_generation',
        'fact_market',
        'bridge_facility_mapping',
    ]
    for t in all_tables:
        try:
            n = conn.execute(f'SELECT COUNT(*) FROM {t}').fetchone()[0]
            print(f'  {t:<35} {n:>10,}')
        except Exception as e:
            print(f'  {t:<35} ERROR: {e}')

    # 2. Primary key uniqueness
    print('\n=== Primary key uniqueness (0 = OK) ===')
    pk_checks = [
        ('dim_ass1_facilities.ass1_facility_id',
         'SELECT COUNT(*) - COUNT(DISTINCT ass1_facility_id) FROM dim_ass1_facilities'),
        ('dim_ass1_facilities.source_facility_key',
         'SELECT COUNT(*) - COUNT(DISTINCT source_facility_key) FROM dim_ass1_facilities'),
        ('fact_nger_annual',
         "SELECT COUNT(*) - COUNT(DISTINCT ass1_facility_id::TEXT||'|'||year::TEXT)"
         ' FROM fact_nger_annual'),
        ('fact_abs_state_year',
         "SELECT COUNT(*) - COUNT(DISTINCT state||'|'||year::TEXT) FROM fact_abs_state_year"),
        ('fact_realtime_generation',
         "SELECT COUNT(*) - COUNT(DISTINCT timestamp_utc::TEXT||'|'||facility_code)"
         ' FROM fact_realtime_generation'),
        ('fact_market',
         "SELECT COUNT(*) - COUNT(DISTINCT timestamp_utc::TEXT||'|'||network_region)"
         ' FROM fact_market'),
    ]
    for label, sql in pk_checks:
        dupes = conn.execute(sql).fetchone()[0]
        print(f'  {label:<45} {"OK" if dupes == 0 else f"WARN: {dupes} dupes"}')

    # 3. FK NULL rates
    print('\n=== FK NULL rates (0% = OK) ===')
    fk_checks = [
        ('fact_nger_annual.ass1_facility_id',
         'SELECT ROUND(100.0*SUM(CASE WHEN ass1_facility_id IS NULL THEN 1 ELSE 0 END)/COUNT(*),2)'
         ' FROM fact_nger_annual'),
        ('fact_realtime_generation.facility_code',
         'SELECT ROUND(100.0*SUM(CASE WHEN facility_code IS NULL THEN 1 ELSE 0 END)/COUNT(*),2)'
         ' FROM fact_realtime_generation'),
    ]
    for label, sql in fk_checks:
        pct = conn.execute(sql).fetchone()[0]
        print(f'  {label:<45} {pct}% null')


    # 4. Orphan FK check
    print(chr(10) + "=== Orphan FK check (0 = OK) ===")
    orphan_checks = [
        ("fact_nger_annual.ass1_facility_id -> dim_ass1_facilities",
         "SELECT COUNT(*) FROM fact_nger_annual f"
         " WHERE NOT EXISTS (SELECT 1 FROM dim_ass1_facilities d WHERE d.ass1_facility_id = f.ass1_facility_id)"),
        ("fact_realtime_generation.facility_code -> dim_oe_facilities",
         "SELECT COUNT(*) FROM fact_realtime_generation f"
         " WHERE NOT EXISTS (SELECT 1 FROM dim_oe_facilities d WHERE d.facility_code = f.facility_code)"),
        ("bridge_facility_mapping.oe_facility_code -> dim_oe_facilities",
         "SELECT COUNT(*) FROM bridge_facility_mapping b"
         " WHERE NOT EXISTS (SELECT 1 FROM dim_oe_facilities d WHERE d.facility_code = b.oe_facility_code)"),
        ("bridge_facility_mapping.ass1_facility_id -> dim_ass1_facilities",
         "SELECT COUNT(*) FROM bridge_facility_mapping b"
         " WHERE NOT EXISTS (SELECT 1 FROM dim_ass1_facilities d WHERE d.ass1_facility_id = b.ass1_facility_id)"),
    ]
    for label, sql in orphan_checks:
        orphans = conn.execute(sql).fetchone()[0]
        status = "OK" if orphans == 0 else f"WARN: {orphans} orphans"
        print(f"  {label:<57} {status}")

    # 5. Bridge coverage
    print('\n=== Bridge table coverage ===')
    bridge_stats = conn.execute('''
        SELECT
            COUNT(DISTINCT oe.facility_code)           AS total_oe,
            COUNT(DISTINCT b.oe_facility_code)         AS matched_oe,
            ROUND(
                100.0 * COUNT(DISTINCT b.oe_facility_code)
                / NULLIF(COUNT(DISTINCT oe.facility_code), 0), 1
            )                                          AS match_pct_all,
            SUM(CASE
                WHEN oe.fueltech_id ILIKE '%wind%'
                  OR oe.fueltech_id ILIKE '%solar%'
                  OR oe.fueltech_id ILIKE '%hydro%'
                  OR oe.fueltech_id ILIKE '%biomass%'
                THEN 1 ELSE 0 END)                     AS renewable_oe_count
        FROM dim_oe_facilities oe
        LEFT JOIN bridge_facility_mapping b ON oe.facility_code = b.oe_facility_code
    ''').df()
    display(bridge_stats)

    # 6. Sample cross-dataset query
    print('\n=== Sample: real-time + NGER historical + market (top 5 by confidence) ===')
    sample = conn.execute('''
        SELECT
            oe.facility_code,
            oe.facility_name                           AS oe_name,
            a1.facility_name                           AS ass1_name,
            a1.state,
            bm.confidence,
            (
                SELECT power_mw FROM fact_realtime_generation g
                WHERE g.facility_code = oe.facility_code
                ORDER BY g.timestamp_utc DESC LIMIT 1
            )                                          AS latest_power_mw,
            (
                SELECT emissions_total FROM fact_nger_annual n
                WHERE n.ass1_facility_id = bm.ass1_facility_id
                ORDER BY n.year DESC LIMIT 1
            )                                          AS nger_latest_emissions
        FROM bridge_facility_mapping bm
        JOIN dim_oe_facilities   oe ON bm.oe_facility_code = oe.facility_code
        JOIN dim_ass1_facilities a1 ON bm.ass1_facility_id = a1.ass1_facility_id
        ORDER BY bm.confidence DESC
        LIMIT 5
    ''').df()
    display(sample)

    conn.close()
    print('\nTask 4 validation complete.')
else:
    print('Validation skipped.')


## 15. Task 5: MQTT Subscription and Live Dashboard

This section defines the MQTT subscriber and live dashboard. It subscribes to the Task 3 / Task 6 MQTT stream, decodes the JSON payload, keeps the latest event per facility in memory, upserts received power/emissions events into the Task 4 DuckDB schema, and renders a live Dash Leaflet map.

The standalone `dashboard.py` file remains useful for final demonstration because it can run as a long-lived process. The notebook version below uses the same topic, payload schema, and DuckDB table contract as Task 3 and Task 4.


In [ ]:
# Task 5 configuration and imports.
import json
import os
import threading
import logging
import socket
import time
from pathlib import Path

import pandas as pd
import duckdb
import paho.mqtt.client as mqtt

try:
    import dash
    from dash import dcc, html, Input, Output
    import dash_leaflet as dl
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Task 5 requires dash and dash-leaflet. Install from requirements.txt before running the dashboard."
    ) from exc

TASK5_MQTT_HOST = os.getenv('MQTT_HOST', MQTT_HOST if 'MQTT_HOST' in globals() else 'localhost')
TASK5_MQTT_PORT = int(os.getenv('MQTT_PORT', str(MQTT_PORT if 'MQTT_PORT' in globals() else 1883)))
TASK5_MQTT_TOPIC = os.getenv('MQTT_TOPIC', MQTT_TOPIC if 'MQTT_TOPIC' in globals() else 'comp5339/assignment2/nem/facility')
TASK5_DB_PATH = Path(os.getenv('TASK4_DB', str(TASK4_DB_PATH)))
TASK5_ENABLE_DB_UPSERT = os.getenv('ENABLE_DB_UPSERT', 'false').lower() == 'true'
TASK5_DASHBOARD_PORT = int(os.getenv('DASHBOARD_PORT', '8050'))
TASK5_UPDATE_INTERVAL_MS = 2000

TASK5_REQUIRED_PAYLOAD_FIELDS = [
    'timestamp_utc', 'facility_code', 'facility_name', 'network_region',
    'fueltech_id', 'latitude', 'longitude', 'power_mw', 'emissions_tco2e',
]

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
task5_log = logging.getLogger('task5_dashboard')
print('Task 5 configuration loaded:', TASK5_MQTT_HOST, TASK5_MQTT_PORT, TASK5_MQTT_TOPIC, TASK5_DB_PATH, 'db_upsert=', TASK5_ENABLE_DB_UPSERT)


### 15.1 Subscriber State, MQTT Decode, and DuckDB Upsert

The subscriber keeps only the latest record per facility for visualisation. Every decoded MQTT event is also upserted into `fact_realtime_generation`, using the same `(timestamp_utc, facility_code)` primary key created in Task 4.


In [ ]:
# ── 1. Subscriber shared state (in-memory, thread-safe) ─────────────────────
task5_state_lock = threading.Lock()
task5_latest_by_facility: dict[str, dict] = {}
task5_latest_market: dict[str, dict] = {}
task5_msg_count = [0]
task5_upsert_count = [0]
task5_db_conn: duckdb.DuckDBPyConnection | None = None
task5_db_lock = threading.Lock()
task5_subscriber_stop_event = threading.Event()
task5_mqtt_client = None



# ── 2. MQTT payload decode, normalise, and in-memory state update ──────────────
def task5_normalise_record(record: dict) -> dict | None:
    """Validate and coerce numeric fields in an MQTT payload dict; return None if required fields are missing."""
    missing = [field for field in TASK5_REQUIRED_PAYLOAD_FIELDS if field not in record]
    if missing:
        task5_log.warning('Skipping MQTT payload with missing fields: %s', missing)
        return None
    for field in ('power_mw', 'emissions_tco2e', 'price_aud_per_mwh', 'demand_mw', 'latitude', 'longitude', 'capacity_mw', 'unit_count'):
        value = record.get(field)
        if value is not None:
            try:
                record[field] = float(value)
            except (TypeError, ValueError):
                record[field] = None
    return record


def task5_update_state(record: dict) -> None:
    """Store the latest record for a facility and update regional price/demand state under the state lock."""
    facility_code = record.get('facility_code')
    if not facility_code:
        return
    with task5_state_lock:
        task5_latest_by_facility[facility_code] = record
        region = record.get('network_region')
        price = record.get('price_aud_per_mwh')
        demand = record.get('demand_mw')
        if region and (price is not None or demand is not None):
            task5_latest_market[region] = {'price': price, 'demand': demand, 'timestamp_utc': record.get('timestamp_utc', '')}
        task5_msg_count[0] += 1


def task5_snapshot() -> tuple[dict, dict, int, int]:
    """Return a thread-safe copy of the current facility state dict."""
    with task5_state_lock:
        return dict(task5_latest_by_facility), dict(task5_latest_market), int(task5_msg_count[0]), int(task5_upsert_count[0])



# ── 3. DuckDB upsert helper (writes live records into the Task 4 schema) ────────
def task5_open_task4_db() -> duckdb.DuckDBPyConnection | None:
    """Open (or reuse) the Task 4 DuckDB connection for subscriber persistence."""
    if not TASK5_DB_PATH.exists():
        task5_log.warning('Task 4 DB not found at %s; DB integration skipped.', TASK5_DB_PATH)
        return None
    try:
        return duckdb.connect(str(TASK5_DB_PATH))
    except Exception as exc:
        task5_log.debug('Task 5 DB open skipped: %s', exc)
        return None


def task5_upsert_record(record: dict) -> None:
    """Insert or replace a real-time generation record in the Task 4 DuckDB table."""
    if not TASK5_ENABLE_DB_UPSERT:
        return
    timestamp_utc = record.get('timestamp_utc')
    facility_code = record.get('facility_code')
    if not timestamp_utc or not facility_code:
        return
    with task5_db_lock:
        conn = task5_open_task4_db()
        if conn is None:
            return
        try:
            conn.execute(
                '''
                INSERT INTO fact_realtime_generation
                    (timestamp_utc, facility_code, power_mw, emissions_tco2e)
                SELECT ?, ?, ?, ?
                WHERE EXISTS (SELECT 1 FROM dim_oe_facilities WHERE facility_code = ?)
                ON CONFLICT (timestamp_utc, facility_code)
                DO UPDATE SET power_mw = excluded.power_mw,
                              emissions_tco2e = excluded.emissions_tco2e
                ''',
                [timestamp_utc, facility_code, record.get('power_mw'), record.get('emissions_tco2e'), facility_code],
            )
            with task5_state_lock:
                task5_upsert_count[0] += 1
        except Exception as exc:
            task5_log.debug('Task 5 DB upsert skipped: %s', exc)
        finally:
            conn.close()



# ── 4. Paho MQTT callbacks and background subscriber thread ─────────────────────
def task5_on_connect(client, userdata, connect_flags, reason_code, properties=None):
    """Paho on_connect callback: subscribe to the MQTT topic when the connection is established."""
    if str(reason_code) in ('0', 'Success'):
        task5_log.info('Task 5 MQTT connected. Subscribing to %s', TASK5_MQTT_TOPIC)
        client.subscribe(TASK5_MQTT_TOPIC, qos=1)
    else:
        task5_log.error('Task 5 MQTT connect failed: %s', reason_code)


def task5_on_message(client, userdata, msg):
    """Paho on_message callback: decode JSON payload, normalise, update in-memory state, and persist to DB."""
    try:
        record = json.loads(msg.payload.decode('utf-8'))
    except (json.JSONDecodeError, UnicodeDecodeError) as exc:
        task5_log.warning('Failed to decode MQTT payload: %s', exc)
        return
    record = task5_normalise_record(record)
    if record is None:
        return
    task5_update_state(record)
    task5_upsert_record(record)


def task5_create_mqtt_client() -> mqtt.Client:
    """Create and configure a Paho MQTT client with the Task 5 callbacks attached."""
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = task5_on_connect
    client.on_message = task5_on_message
    return client


def task5_start_subscriber_in_background() -> mqtt.Client:
    """Launch the MQTT subscriber loop in a background daemon thread."""
    task5_subscriber_stop_event.clear()
    client = task5_create_mqtt_client()

    def _loop():
        """Background thread target: connect and run the Paho network loop indefinitely, retrying on failure."""
        while not task5_subscriber_stop_event.is_set():
            try:
                client.connect(TASK5_MQTT_HOST, TASK5_MQTT_PORT, keepalive=60)
                client.loop_forever()
            except OSError as exc:
                if not task5_subscriber_stop_event.is_set():
                    task5_log.warning('Task 5 MQTT connection failed (%s); retrying in 5s', exc)
                    time.sleep(5)
            finally:
                try:
                    client.disconnect()
                except Exception:
                    pass
    threading.Thread(target=_loop, daemon=True, name='task5-mqtt-sub').start()
    return client


def task5_stop_subscriber() -> None:
    """Signal the background subscriber thread to stop and disconnect the MQTT client."""
    global task5_db_conn, task5_mqtt_client
    task5_subscriber_stop_event.set()
    if task5_mqtt_client is not None:
        try:
            task5_mqtt_client.disconnect()
            print('Task 5 MQTT subscriber stopped.')
        except Exception as exc:
            print(f'Skipped Task 5 MQTT disconnect: {exc}')
        task5_mqtt_client = None
    with task5_db_lock:
        if task5_db_conn is not None:
            try:
                task5_db_conn.close()
                print('Task 5 DuckDB connection closed.')
            except Exception as exc:
                print(f'Skipped Task 5 DuckDB close: {exc}')
            task5_db_conn = None



# ── 5. Subscriber startup toggle ────────────────────────────────────────────────
RUN_TASK5_SUBSCRIBER = False
if RUN_TASK5_SUBSCRIBER:
    task5_mqtt_client = task5_start_subscriber_in_background()
    print('Task 5 subscriber started in background.')
else:
    print('Task 5 subscriber not started. Set RUN_TASK5_SUBSCRIBER = True for a live demo.')


### 15.2 Dash Leaflet Dashboard

The dashboard renders live markers from the subscriber state. Marker popups include facility name, fuel type, latest power, emissions, price, and demand. The controls provide the required power/emissions selector plus optional region and fuel filters.


In [ ]:
TASK5_FUELTECH_COLOUR = {
    'coal_black': '#374151', 'coal_brown': '#78350F', 'gas_ccgt': '#EF4444', 'gas_ocgt': '#F97316',
    'gas_recip': '#FB923C', 'gas_steam': '#FDBA74', 'gas_wcmg': '#FDE68A', 'hydro': '#06B6D4',
    'pumps': '#0891B2', 'solar_utility': '#F59E0B', 'wind': '#3B82F6', 'bioenergy_biomass': '#22C55E',
    'bioenergy_biogas': '#16A34A', 'landfill_gas': '#84CC16', 'distillate': '#DC2626', 'mixed': '#8B5CF6',
}
TASK5_DEFAULT_COLOUR = '#9CA3AF'
TASK5_FUELTECH_LABEL = {'coal_black': 'Black Coal', 'coal_brown': 'Brown Coal', 'gas_ccgt': 'Gas (CCGT)', 'gas_ocgt': 'Gas (OCGT)', 'gas_recip': 'Gas (Recip)', 'gas_steam': 'Gas (Steam)', 'gas_wcmg': 'Gas (WCMG)', 'hydro': 'Hydro', 'pumps': 'Pumped Hydro', 'solar_utility': 'Solar', 'wind': 'Wind', 'bioenergy_biomass': 'Biomass', 'bioenergy_biogas': 'Biogas', 'landfill_gas': 'Landfill Gas', 'distillate': 'Distillate', 'mixed': 'Mixed'}
TASK5_ALL_REGIONS = ['All', 'NSW1', 'QLD1', 'VIC1', 'SA1', 'TAS1']
TASK5_ALL_FUELTECHS = ['All'] + sorted(set(TASK5_FUELTECH_LABEL.values()))


def task5_make_markers(facilities: dict, view_mode: str, region_filter: str, fueltech_filter: str) -> list:
    """Build a list of Dash-Leaflet CircleMarker components from the current facility snapshot."""
    markers = []
    for facility_code, record in facilities.items():
        region = record.get('network_region', '')
        fueltech_id = record.get('fueltech_id', '')
        fueltech_label = TASK5_FUELTECH_LABEL.get(fueltech_id, fueltech_id)
        if region_filter != 'All' and region != region_filter:
            continue
        if fueltech_filter != 'All' and fueltech_label != fueltech_filter:
            continue
        lat = record.get('latitude')
        lon = record.get('longitude')
        if lat is None or lon is None:
            continue
        power = record.get('power_mw') or 0.0
        emissions = record.get('emissions_tco2e') or 0.0
        display_value = power if view_mode == 'power' else emissions
        display_unit = 'MW' if view_mode == 'power' else 'tCO2e'
        colour = TASK5_FUELTECH_COLOUR.get(fueltech_id, TASK5_DEFAULT_COLOUR)
        price = record.get('price_aud_per_mwh')
        demand = record.get('demand_mw')
        popup = html.Div([
            html.B(record.get('facility_name', facility_code)), html.Br(),
            html.Span(f'Code: {facility_code}'), html.Br(), html.Br(),
            html.B('Region: '), html.Span(region), html.Br(),
            html.B('Type: '), html.Span(fueltech_label), html.Br(),
            html.B('Power: '), html.Span(f'{power:.2f} MW'), html.Br(),
            html.B('Emissions: '), html.Span(f'{emissions:.4f} tCO2e'), html.Br(),
            html.B('Price: '), html.Span(f"${price:.2f}/MWh" if price is not None else 'N/A'), html.Br(),
            html.B('Demand: '), html.Span(f'{demand:.0f} MW' if demand is not None else 'N/A'), html.Br(), html.Br(),
            html.Small(f"Event: {record.get('timestamp_local') or record.get('timestamp_utc', '')}"),
        ], style={'fontFamily': 'monospace', 'fontSize': '12px', 'minWidth': '220px'})
        markers.append(dl.CircleMarker(center=[lat, lon], radius=max(5, min(28, 5 + power / 60)), color=colour, fillColor=colour, fillOpacity=0.8, opacity=0.95, weight=1, children=[dl.Tooltip(f"{record.get('facility_name', facility_code)}: {display_value:.1f} {display_unit}"), dl.Popup(popup)]))
    return markers


def task5_create_dash_app() -> dash.Dash:
    """Construct and return the Dash application layout with the map and control widgets."""
    app = dash.Dash(__name__, title='COMP5339 NEM Live Dashboard', suppress_callback_exceptions=True)
    app.layout = html.Div([
        dcc.Interval(id='task5_tick', interval=TASK5_UPDATE_INTERVAL_MS, n_intervals=0),
        html.Div([html.Div('NEM Live Electricity Dashboard', style={'fontWeight': '700', 'fontSize': '18px'}), html.Div([html.Span('MSG '), html.Span(id='task5_hdr_msg', children='0'), html.Span('  FAC '), html.Span(id='task5_hdr_fac', children='0'), html.Span('  DB '), html.Span(id='task5_hdr_db', children='0')], style={'fontFamily': 'monospace', 'fontSize': '12px'})], style={'height': '42px', 'padding': '10px 16px', 'display': 'flex', 'justifyContent': 'space-between', 'alignItems': 'center', 'background': '#0F172A', 'color': '#F9FAFB'}),
        html.Div(id='task5_market_bar', style={'minHeight': '32px', 'padding': '7px 16px', 'background': '#111827', 'color': '#D1D5DB', 'fontFamily': 'monospace', 'fontSize': '12px'}),
        html.Div([html.Div([html.Label('View Mode'), dcc.RadioItems(id='task5_view_mode', options=[{'label': ' Power (MW)', 'value': 'power'}, {'label': ' Emissions (tCO2e)', 'value': 'emissions'}], value='power'), html.Br(), html.Label('Network Region'), dcc.Dropdown(id='task5_region_filter', options=[{'label': r, 'value': r} for r in TASK5_ALL_REGIONS], value='All', clearable=False), html.Br(), html.Label('Fuel Technology'), dcc.Dropdown(id='task5_fueltech_filter', options=[{'label': f, 'value': f} for f in TASK5_ALL_FUELTECHS], value='All', clearable=False), html.Hr(), html.Div(id='task5_live_stats')], style={'width': '260px', 'padding': '14px', 'background': '#1F2937', 'color': '#E5E7EB', 'fontFamily': 'monospace', 'fontSize': '12px'}), dl.Map(id='task5_map', center=[-27.0, 134.0], zoom=5, children=[dl.TileLayer(url='https://{s}.basemaps.cartocdn.com/dark_all/{z}/{x}/{y}{r}.png', attribution='CARTO'), dl.LayerGroup(id='task5_markers_layer')], style={'flex': '1', 'height': 'calc(100vh - 74px)'})], style={'display': 'flex'}),
    ])

    @app.callback(Output('task5_markers_layer', 'children'), Output('task5_hdr_msg', 'children'), Output('task5_hdr_fac', 'children'), Output('task5_hdr_db', 'children'), Output('task5_market_bar', 'children'), Output('task5_live_stats', 'children'), Input('task5_tick', 'n_intervals'), Input('task5_view_mode', 'value'), Input('task5_region_filter', 'value'), Input('task5_fueltech_filter', 'value'))
    def task5_refresh(_n, view_mode, region_filter, fueltech_filter):
        """Dash callback: refresh map markers and market-stats display on every interval tick."""
        facilities, market, msg_count, upsert_count = task5_snapshot()
        markers = task5_make_markers(facilities, view_mode, region_filter, fueltech_filter)
        market_items = []
        for region in ['NSW1', 'QLD1', 'VIC1', 'SA1', 'TAS1']:
            snap = market.get(region)
            if snap:
                market_items.append(html.Span(f"{region}: ${snap.get('price'):.2f}/MWh | {snap.get('demand'):.0f} MW  "))
        if not market_items:
            market_items = [html.Span('Waiting for market data...')]
        visible = [r for r in facilities.values() if region_filter == 'All' or r.get('network_region') == region_filter]
        total_power = sum((r.get('power_mw') or 0.0) for r in visible)
        total_emissions = sum((r.get('emissions_tco2e') or 0.0) for r in visible)
        stats = [html.Div(f'Generation: {total_power:,.0f} MW'), html.Div(f'Emissions: {total_emissions:,.2f} tCO2e'), html.Div(f'Visible facilities: {len(visible)}')]
        return markers, str(msg_count), str(len(facilities)), str(upsert_count), market_items, stats
    return app


def task5_run_dashboard() -> None:
    """Start the Dash development server for the Task 5 live dashboard."""
    app = task5_create_dash_app()
    app.run(debug=False, host='127.0.0.1', port=TASK5_DASHBOARD_PORT)


def validate_task5_runtime_state() -> None:
    """Raise if the prerequisite runtime state (subscriber thread, DB) is not ready."""
    facilities, market, msg_count, upsert_count = task5_snapshot()
    print('MQTT messages decoded:', msg_count)
    print('Facilities currently on dashboard:', len(facilities))
    print('Market regions currently shown:', sorted(market.keys()))
    print('DuckDB upsert attempts:', upsert_count)

print('Task 5 dashboard helpers ready. Call task5_run_dashboard() for a live dashboard.')


## 16. Task 6: Continuous Execution Publisher

Task 6 simulates an unbounded real-time stream without repeatedly calling the OpenElectricity API. Task 1 retrieves at least one week of source data once, Task 2 consolidates it, and Task 4 stores the cleaned streamable records in DuckDB. Task 6 then repeatedly pulls a bounded chunk from that local database, publishes the chunk through the Task 3 MQTT contract, waits 60 seconds, and continues with the next chunk.

This matches the lecturer's clarification:

- no continuous API calls after the initial Task 1 retrieval
- each round publishes a bounded chunk, default `3,000` records
- the loop waits `60` seconds before the next round
- when the local dataset is exhausted, the state resets and the publisher replays from the beginning to simulate an unbounded stream

The standalone `publisher.py` can still be used, but this notebook cell is the clearest version for assessment because the chunk size and 60-second delay are explicit.


In [ ]:
# Task 6 chunk-based continuous execution.
# The stream is simulated from local DuckDB/CSV data, not from repeated API calls.

TASK6_DB_PATH = TASK4_DB_PATH
TASK6_CSV_PATH = TASK2_CONSOLIDATED_PATH
TASK6_STATE_PATH = TASK3_STATE_PATH
TASK6_MQTT_HOST = os.getenv('MQTT_HOST', MQTT_HOST if 'MQTT_HOST' in globals() else 'localhost')
TASK6_MQTT_PORT = int(os.getenv('MQTT_PORT', str(MQTT_PORT if 'MQTT_PORT' in globals() else 1883)))
TASK6_MQTT_TOPIC = os.getenv('MQTT_TOPIC', MQTT_TOPIC if 'MQTT_TOPIC' in globals() else 'comp5339/assignment2/nem/facility')
TASK6_PUBLISH_DELAY_SECONDS = MQTT_MESSAGE_DELAY_SECONDS if 'MQTT_MESSAGE_DELAY_SECONDS' in globals() else 0.1
TASK6_LOOP_DELAY_SECONDS = 60
TASK6_CHUNK_SIZE = int(os.getenv('TASK6_CHUNK_SIZE', '3000'))
TASK6_REPLAY_WHEN_EXHAUSTED = True

print('Task 6 configuration loaded:')
print('  DB source:', TASK6_DB_PATH)
print('  CSV fallback:', TASK6_CSV_PATH)
print('  MQTT topic:', TASK6_MQTT_TOPIC)
print('  chunk size:', TASK6_CHUNK_SIZE)
print('  round delay:', TASK6_LOOP_DELAY_SECONDS, 'seconds')


def task6_default_state() -> dict:
    """Return the initial publisher state dict with zeroed counters."""
    return {
        'last_published_timestamp_utc': None,
        'last_published_facility_code': None,
        'records_published': 0,
    }


def task6_format_stream_chunk(df: pd.DataFrame, source_label: str) -> pd.DataFrame:
    """Convert a DataFrame chunk to a list of JSON-serialisable payload dicts."""
    if df.empty:
        return df
    df = df.copy()
    df['timestamp_utc'] = pd.to_datetime(df['timestamp_utc'], utc=True)
    df['timestamp_local'] = df['timestamp_utc'].dt.tz_convert('Australia/Sydney').dt.strftime('%Y-%m-%dT%H:%M:%S%z')
    df['timestamp_utc'] = df['timestamp_utc'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    df['source'] = source_label
    for column in TASK3_REQUIRED_COLUMNS:
        if column not in df.columns:
            df[column] = None
    df = df[TASK3_REQUIRED_COLUMNS].copy()
    validate_task3_input_columns(df)
    return df


def task6_load_chunk_from_database(state: dict, chunk_size: int = TASK6_CHUNK_SIZE) -> pd.DataFrame:
    """Query the next chunk of records from DuckDB using cursor-based pagination."""
    if not TASK6_DB_PATH.exists():
        raise FileNotFoundError(f'Task 6 DuckDB source does not exist: {TASK6_DB_PATH}')

    # Avoid same-kernel DuckDB configuration errors without stopping the Task 5
    # MQTT subscriber. The subscriber can keep receiving messages while its DB
    # handle is reopened lazily on the next upsert if needed.
    for _duckdb_conn_name in ('task5_db_conn', 'conn', '_conn_ro'):
        _duckdb_conn = globals().get(_duckdb_conn_name)
        if _duckdb_conn is not None:
            try:
                _duckdb_conn.close()
                print(f'Closed stale DuckDB connection: {_duckdb_conn_name}')
            except Exception as exc:
                print(f'Skipped stale DuckDB connection {_duckdb_conn_name}: {exc}')
            globals()[_duckdb_conn_name] = None

    last_ts = state.get('last_published_timestamp_utc')
    last_facility = state.get('last_published_facility_code') or ''
    sql = '''
        SELECT
            g.timestamp_utc,
            g.facility_code,
            f.facility_name,
            f.network_region,
            f.fueltech_id,
            f.latitude,
            f.longitude,
            f.capacity_mw,
            f.unit_count,
            g.power_mw,
            g.emissions_tco2e,
            m.price_aud_per_mwh,
            m.demand_mw
        FROM fact_realtime_generation g
        JOIN dim_oe_facilities f
          ON g.facility_code = f.facility_code
        LEFT JOIN fact_market m
          ON g.timestamp_utc = m.timestamp_utc
         AND f.network_region = m.network_region
        WHERE
            (? IS NULL)
            OR g.timestamp_utc > CAST(? AS TIMESTAMPTZ)
            OR (g.timestamp_utc = CAST(? AS TIMESTAMPTZ) AND g.facility_code > ?)
        ORDER BY g.timestamp_utc, g.facility_code
        LIMIT ?
    '''
    conn = duckdb.connect(str(TASK6_DB_PATH), read_only=True)
    try:
        df = conn.execute(sql, [last_ts, last_ts, last_ts, last_facility, chunk_size]).df()
    finally:
        conn.close()
    return task6_format_stream_chunk(df, source_label='Task6 DuckDB chunk replay')


def task6_load_chunk_from_csv(state: dict, chunk_size: int = TASK6_CHUNK_SIZE) -> pd.DataFrame:
    """Load the next chunk of records from the consolidated CSV file."""
    if not TASK6_CSV_PATH.exists():
        raise FileNotFoundError(f'Task 6 CSV fallback does not exist: {TASK6_CSV_PATH}')
    df = pd.read_csv(TASK6_CSV_PATH)
    selected = prepare_task3_records(df, state=state, only_new_records=True).head(chunk_size)
    return task6_format_stream_chunk(selected, source_label='Task6 CSV chunk replay')


def task6_load_next_chunk(state: dict, chunk_size: int = TASK6_CHUNK_SIZE) -> pd.DataFrame:
    """Load the next data chunk, preferring DuckDB and falling back to CSV."""
    try:
        return task6_load_chunk_from_database(state, chunk_size=chunk_size)
    except Exception as exc:
        print('Task 6 database chunk load failed; falling back to Task 2 CSV cache:', exc)
        return task6_load_chunk_from_csv(state, chunk_size=chunk_size)


def task6_connect_mqtt_client() -> mqtt.Client:
    """Create, configure, and connect the Paho MQTT client for Task 6 publishing."""
    client = create_mqtt_client()
    while True:
        try:
            connect_mqtt_client(client, host=TASK6_MQTT_HOST, port=TASK6_MQTT_PORT, keepalive=MQTT_KEEPALIVE)
            return client
        except OSError as exc:
            print(f'Task 6 MQTT connection failed ({exc}); retrying in 5 seconds...')
            time.sleep(5)


def task6_publish_records(records_df: pd.DataFrame, client: mqtt.Client, state: dict) -> dict:
    """Publish all records in a DataFrame chunk to MQTT with the required inter-message delay."""
    if records_df.empty:
        print('Task 6: no records to publish in this round.')
        return state

    print(f'Task 6: publishing chunk of {len(records_df):,} records...')
    for _, row in records_df.iterrows():
        payload = record_to_payload(row.to_dict())
        message = json.dumps(payload, allow_nan=False)
        info = client.publish(TASK6_MQTT_TOPIC, message, qos=1)
        info.wait_for_publish(timeout=5)
        if info.rc != mqtt.MQTT_ERR_SUCCESS:
            print(f"WARNING: Task 6 MQTT publish failed for {payload['facility_code']} at {payload['timestamp_utc']} with rc={info.rc} -- skipping record")
            continue
        state['last_published_timestamp_utc'] = payload['timestamp_utc']
        state['last_published_facility_code'] = payload['facility_code']
        state['records_published'] = int(state.get('records_published', 0)) + 1
        save_publish_state(state, TASK6_STATE_PATH)
        time.sleep(TASK6_PUBLISH_DELAY_SECONDS)
    return state


def run_task6_continuous(max_rounds: int | None = None, chunk_size: int = TASK6_CHUNK_SIZE) -> None:
    """Run the continuous Task 6 loop: load chunk, publish, sleep 60 s, repeat indefinitely."""
    state = load_publish_state(TASK6_STATE_PATH)
    client = task6_connect_mqtt_client()
    round_number = 0
    try:
        while True:
            round_number += 1
            print(f'=== Task 6 round {round_number}: next {chunk_size:,} records ===')
            selected = task6_load_next_chunk(state, chunk_size=chunk_size)

            if selected.empty and TASK6_REPLAY_WHEN_EXHAUSTED:
                print('Task 6 reached the end of the local dataset; resetting state to simulate an unbounded stream.')
                state = task6_default_state()
                save_publish_state(state, TASK6_STATE_PATH)
                selected = task6_load_next_chunk(state, chunk_size=chunk_size)

            state = task6_publish_records(selected, client, state)
            save_publish_state(state, TASK6_STATE_PATH)
            print('Task 6 state:', state)

            if max_rounds is not None and round_number >= max_rounds:
                break

            print(f'Task 6 sleeping {TASK6_LOOP_DELAY_SECONDS} seconds before next round...')
            time.sleep(TASK6_LOOP_DELAY_SECONDS)
    finally:
        client.loop_stop()
        client.disconnect()
        print('Task 6 publisher stopped cleanly.')


def run_task6_demo_publish(limit: int = 300, reset_state_first: bool = True) -> dict:
    """Publish a single demonstration chunk and exit (used when RUN_TASK6_CONTINUOUS=False)."""
    if reset_state_first:
        reset_publish_state(TASK6_STATE_PATH)
    state = load_publish_state(TASK6_STATE_PATH)
    selected = task6_load_next_chunk(state, chunk_size=limit)
    client = task6_connect_mqtt_client()
    try:
        state = task6_publish_records(selected, client, state)
        save_publish_state(state, TASK6_STATE_PATH)
        return state
    finally:
        client.loop_stop()
        client.disconnect()


In [ ]:
# Use a second notebook kernel to publish while this server is running.
RUN_TASK5_DASHBOARD = False
if RUN_TASK5_DASHBOARD:
    print(f"Starting Dash Leaflet at http://127.0.0.1:{TASK5_DASHBOARD_PORT}/")
    task5_run_dashboard()


In [ ]:
# ── Task 6 Execution ─────────────────────────────────────────────────────────
# RUN_TASK6_DEMO : 2 rounds x 300 records (approx 2 min total). Shows:
#   - multi-round continuous execution with the required 60-second round delay
#   - cursor-based 'only new records' pagination (Round 2 starts where Round 1 ended)
#   Use this setting for assessment / grading.
# RUN_TASK6_FULL : Unbounded production stream. Requires Kernel > Interrupt to stop.

RUN_TASK6_DEMO = False    # bounded demo: 2 rounds x 300 records
RUN_TASK6_FULL = False   # production: unbounded stream

if RUN_TASK6_DEMO:
    # chunk_size=300 keeps each round short (~30 s) while proving cursor advance:
    # Round 2 starts from where Round 1 left off, so only new records are published.
    run_task6_continuous(max_rounds=2, chunk_size=300)
elif RUN_TASK6_FULL:
    run_task6_continuous(chunk_size=TASK6_CHUNK_SIZE)
else:
    print('Task 6 not started.')
    print('  Grading demo (2 rounds, ~2 min): set RUN_TASK6_DEMO = True')
    print('  Production stream (unbounded):   set RUN_TASK6_FULL = True')

In [ ]:
# Optional integration check. Run after creating the Task 4 DB and starting
# the subscriber with a local broker. Uses the saved replay cursor.
import time as _time

RUN_MQTT_INTEGRATION_CHECK = False
if RUN_MQTT_INTEGRATION_CHECK:
    previous_upsert_setting = TASK5_ENABLE_DB_UPSERT
    try:
        # 1. Enable DuckDB upsert for this validation run only
        TASK5_ENABLE_DB_UPSERT = True
        print(f"DuckDB upsert enabled for validation: {TASK5_ENABLE_DB_UPSERT}")

        # 2. Zero counters so the result is unambiguous
        with task5_state_lock:
            task5_msg_count[0] = 0
            task5_upsert_count[0] = 0
        print("Counters reset  ->  msg_count=0, upsert_count=0")
        print()

        # 3. Publish 5 records via the Task 6 demo helper
        #    (uses the same Task 3 MQTT topic and payload schema)
        print("Publishing 5 records via run_task6_demo_publish(limit=5) ...")
        run_task6_demo_publish(limit=5, reset_state_first=False)

        # 4. Give the background subscriber thread time to receive and decode
        _time.sleep(2.0)

        # 5. Read and display the subscriber state
        _fac_v, _mkt_v, _msg_v, _ups_v = task5_snapshot()
        print()
        print(f"MQTT messages decoded by subscriber : {_msg_v}")
        print(f"Facilities in live dashboard state  : {len(_fac_v)}")
        print(f"DuckDB upsert attempts              : {_ups_v}")
        print()

        if _msg_v >= 5:
            print("OK  Subscriber correctly decoded all 5 published messages.")
        else:
            print(f"WARN  Subscriber decoded only {_msg_v}/5 -- is the MQTT broker running?")
        if _ups_v >= 1:
            print("OK  Subscriber attempted DuckDB upserts.")
        else:
            print("WARN  DuckDB upsert count is 0 -- check Task 4 DB path and table schema.")
    finally:
        TASK5_ENABLE_DB_UPSERT = previous_upsert_setting


## AI Acknowledgement
We acknowledge the use of ChatGPT (https://chat.openai.com/), using GPT-5, to refine the academic language of my own work. The tool was used for improving academic coding quality and consistency. We completed multiple iterations with the tool in our work. The output from the tool included refined sentences, improved phrasing, and structured academic expressions, and is included in the Abstract, Introduction, and Project Problem sections of my assessment. We
modified the AI-generated content by adapting it to align with my original arguments, ensuring accuracy of references, and maintaining a consistent academic tone, because it was necessary to preserve the integrity, clarity, and correctness of our work.